In [ ]:
# Setup: imports, physical constants, fiducial triple (15+15 Msun inner binary, 30 Msun tertiary), detector/PSD settings, and plot style.

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp, cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.signal import hilbert, savgol_filter
from scipy.special import jv
from scipy.optimize import brentq
import rebound

try:
    from pycbc.psd import aLIGOZeroDetHighPower
    _HAVE_PYCBC = True
except Exception:
    _HAVE_PYCBC = False
    print("WARNING: pycbc not installed. Faithfulness will not run.")
    print("Fix: pip install pycbc")

G     = 6.67430e-11
c     = 2.99792458e8
M_sun = 1.98847e30
AU    = 1.496e11
yr    = 3.156e7
pc    = 3.085677581e16

m1    = 15.0 * M_sun
m2    = 15.0 * M_sun
m3    = 30.0 * M_sun
a_in  = 1.0  * AU
a_out = 30.0 * AU
e_in  = 0.1
e_out = 0.3
i_deg = 65.0
rho_cluster   = 1e5 * M_sun / pc**3
sigma_cluster = 10.0e3
H_hard        = 18.0

R            = 410e6 * pc
f_band_entry = 10.0
f_gw_max     = 1024.0

DT_RESAMPLE = 1e-5
F_LOW       = 10.0
F_HIGH      = 1024.0

COL_W = 3.35
TXT_W = 7.10

plt.rcParams.update({
    'font.family':      'serif',
    'font.serif':       ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size':        8.5,
    'axes.labelsize':   9,
    'axes.titlesize':   8.5,
    'xtick.labelsize':  8,
    'ytick.labelsize':  8,
    'legend.fontsize':  7.2,
    'axes.linewidth':   0.7,
    'lines.linewidth':  1.1,
    'lines.markersize': 3.0,
    'lines.markeredgewidth': 0.6,
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'xtick.major.size': 3.5, 'ytick.major.size': 3.5,
    'xtick.minor.size': 2.0, 'ytick.minor.size': 2.0,
    'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
    'xtick.minor.width': 0.5, 'ytick.minor.width': 0.5,
    'axes.grid': False,
    'legend.frameon': False,
    'legend.handlelength': 1.4,
    'legend.handletextpad': 0.5,
    'legend.labelspacing': 0.3,
    'legend.borderpad': 0.3,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'figure.facecolor': 'white',
    'axes.labelpad': 2.5,
})

C_TRIPLE   = '#22314e'
C_ISO      = '#a23b2e'
C_DECOUPLE = '#2f6b4f'
C_WAVE     = '#b5852a'
SPINE      = '#000000'

def style_ax(ax):
    """ApJ box frame: all four spines, inward major+minor ticks all sides."""
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.7); s.set_color('black')
    ax.tick_params(which='both', top=True, right=True, direction='in')

def _dadt_gw(a, e, m1, m2):
    M  = m1 + m2
    fe = (1.0 + 73.0/24.0*e**2 + 37.0/96.0*e**4) / (1.0 - e**2)**3.5
    return (64.0/5.0) * G**3 * m1 * m2 * M / (c**5 * a**3) * fe

def _dadt_hard(a, rho, sigma, H):
    return H * G * rho * a**2 / sigma

def a_decouple(e, rho=rho_cluster, sigma=sigma_cluster, H=H_hard, m1=m1, m2=m2):
    f = lambda a: _dadt_gw(a, e, m1, m2) - _dadt_hard(a, rho, sigma, H)
    return brentq(f, 1e-8*AU, 10.0*AU, maxiter=200)

M_in  = m1 + m2
M_tot = m1 + m2 + m3
mu_in = m1 * m2 / M_in
eta   = (m1 - m2) / M_in

print("Cell 1 loaded.")
print(f"  Inner binary : m1={m1/M_sun:.0f} + m2={m2/M_sun:.0f} = {M_in/M_sun:.0f} Msun")
print(f"  Tertiary     : m3={m3/M_sun:.0f} Msun  (m3/M_in={m3/M_in:.1f})")
print(f"  a_in={a_in/AU:.1f} AU  e_in={e_in}  a_out={a_out/AU:.0f} AU  e_out={e_out}")
print(f"  i={i_deg} deg  R={R/pc/1e6:.0f} Mpc")
print(f"  eta={eta:.3f} — "
      f"{'pure quadrupole (equal mass)' if abs(eta)<0.01 else 'EKL active'}")
print(f"  F_LOW={F_LOW} Hz  F_HIGH={F_HIGH} Hz  DT_RESAMPLE={DT_RESAMPLE:.0e} s")
print(f"  pycbc PSD available: {_HAVE_PYCBC}")

In [ ]:
# Analytic helpers: KL, GR-precession and Peters GW timescales, decoupling condition, and orbit-averaged Peters da/dt and de/dt.

def T_KL(m1, m2, m3, a_in, a_out, e_out):
    """
    Kozai-Lidov oscillation period (quadrupole, test-particle limit).
    Lidov 1962, Kozai 1962.
    """
    M_in  = m1 + m2
    M_tot = m1 + m2 + m3
    P_in  = 2*math.pi * math.sqrt(a_in**3  / (G * M_in))
    P_out = 2*math.pi * math.sqrt(a_out**3 / (G * M_tot))
    return (P_out**2 / P_in) * (M_tot / m3) * (1 - e_out**2)**1.5

def T_GR(m1, m2, a_in, e_in):
    """
    GR precession timescale (1PN Schwarzschild).
    Precession rate: domega/dt = 3*G*M*n / (c^2*a*(1-e^2))
    T_GR = 2*pi / (domega/dt)
    """
    M_in = m1 + m2
    n    = math.sqrt(G * M_in / a_in**3)
    domega_dt = 3 * G * M_in * n / \
                (c**2 * a_in * (1 - e_in**2))
    return 2 * math.pi / domega_dt

def T_GW(m1, m2, a_in, e_in):
    """
    Peters (1964) GW inspiral timescale.
    t_GW = 12/19 * c0^4/beta * integral(e)
    Approximated as Peters Eq. 5.14.
    """
    M_in = m1 + m2
    beta = 64/5 * G**3 * m1 * m2 * M_in / c**5

    f_e  = (1 + 73/24*e_in**2 + 37/96*e_in**4) / \
           (1 - e_in**2)**3.5
    return a_in**4 / (4 * beta * f_e)

def P_out_current(m1, m2, m3, a_out):
    """Outer orbital period."""
    M_tot = m1 + m2 + m3
    return 2*math.pi * math.sqrt(a_out**3 / (G * M_tot))

def is_decoupled_GW_Pout(m1, m2, m3, a_in, e_in, a_out):
    """
    Condition 1: t_GW < P_out
    Once GW merger is faster than one outer orbit,
    the tertiary cannot torque the inner binary
    before it merges.
    """
    return T_GW(m1, m2, a_in, e_in) < \
           P_out_current(m1, m2, m3, a_out)

def is_decoupled_GW_KL(m1, m2, m3, a_in, e_in, a_out, e_out):
    """
    Condition 2: t_GW < T_KL
    Once GW inspiral is faster than a KL cycle,
    the binary evolves independently.
    Antognini (2015), Eq. 15.
    """
    return T_GW(m1, m2, a_in, e_in) < \
           T_KL(m1, m2, m3, a_in, a_out, e_out)

def gw_frequency(a, m1, m2):
    """Instantaneous GW frequency from semi-major axis."""
    M_in = m1 + m2
    return (1/math.pi) * math.sqrt(G * M_in / a**3)

def a_from_frequency(f_gw, m1, m2):
    """Semi-major axis corresponding to a GW frequency."""
    M_in = m1 + m2
    return (G * M_in / (math.pi * f_gw)**2)**(1/3)

def isco_radius(m1, m2):
    """Schwarzschild ISCO for total mass."""
    return 6.0 * G * (m1 + m2) / c**2

tkl  = T_KL(m1, m2, m3, a_in, a_out, e_out)
tgr  = T_GR(m1, m2, a_in, e_in)
tgw  = T_GW(m1, m2, a_in, e_in)
pout = P_out_current(m1, m2, m3, a_out)

print("Cell 2 loaded: analytic helpers defined.")
print(f"\n── Initial timescales ───────────────────────────────")
print(f"  T_KL  = {tkl/yr:.2f} yr")
print(f"  T_GR  = {tgr/yr:.2f} yr")
print(f"  T_GW  = {tgw/yr:.4e} yr")
print(f"  P_out = {pout/yr:.4f} yr")
print(f"\n  T_KL / T_GR = {tkl/tgr:.3f} "
      f"({'GR quenching active' if tgr < tkl else 'GR quenching weak — EKL may dominate'})")
print(f"  T_GW / T_KL = {tgw/tkl:.3e} "
      f"(decoupling not yet triggered at a_in=1 AU)")

In [ ]:
# Triple builder: constructs the REBOUND/REBOUNDx hierarchical triple with 1PN GR precession and checks stability and timescales.

try:
    import reboundx
    _HAVE_REBOUNDX = True
except ImportError:
    _HAVE_REBOUNDX = False
    print("WARNING: reboundx not installed — GR precession unavailable.")
    print("  Fix: pip install reboundx  (or call build_triple(..., gr=False))")

def build_triple(m1, m2, m3, a_in, e_in, a_out, e_out,
                 i_deg, omega_in=0.0, Omega_in=0.0,
                 omega_out=math.pi/4, f_in=0.0, f_out=0.0,
                 gr=True):
    """
    Build REBOUND hierarchical triple simulation.

    gr : bool (default True)
        Add 1PN GR precession via reboundx gr_full. gr=False → pure Newtonian.

    Returns (sim, rebx). rebx is None when gr=False. Keep rebx in scope for the
    whole integration or the GR force detaches.
    """
    sim = rebound.Simulation()
    sim.units = ('m', 's', 'kg')
    sim.G     = G

    sim.add(m=m1)
    sim.add(m=m2,
            a=a_in, e=e_in,
            inc=0.0,
            Omega=Omega_in,
            omega=omega_in,
            f=f_in,
            primary=sim.particles[0])
    sim.move_to_com()

    sim.add(m=m3,
            a=a_out, e=e_out,
            inc=math.radians(i_deg),
            Omega=0.0,
            omega=omega_out,
            f=f_out)

    sim.move_to_com()
    sim.integrator = "ias15"

    rebx = None
    if gr:
        if not _HAVE_REBOUNDX:
            raise RuntimeError("gr=True but reboundx not installed. "
                               "pip install reboundx, or call with gr=False.")
        rebx = reboundx.Extras(sim)
        gr_force = rebx.load_force("gr_full")
        gr_force.params["c"] = c
        rebx.add_force(gr_force)

    return sim, rebx

sim_test, rebx_test = build_triple(m1, m2, m3, a_in, e_in,
                                   a_out, e_out, i_deg, gr=False)

orb_in  = sim_test.particles[1].orbit(primary=sim_test.particles[0])
orb_out = sim_test.particles[2].orbit()

print("Cell 3 loaded: build_triple() with GR precession.")
print(f"\n── Triple verification (GR on) ──────────────────────")
print(f"  Inner: a={orb_in.a/AU:.4f} AU  "
      f"e={orb_in.e:.4f}  "
      f"inc={math.degrees(orb_in.inc):.2f} deg")
print(f"  Outer: a={orb_out.a/AU:.4f} AU  "
      f"e={orb_out.e:.4f}  "
      f"inc={math.degrees(orb_out.inc):.2f} deg")
print(f"  Outer inc should be ~{i_deg} deg — "
      f"{'OK' if abs(math.degrees(orb_out.inc)-i_deg)<2 else 'CHECK'}")

ratio_actual   = a_out / a_in
_q_out = m3 / M_in
ratio_mardling = (2.8 / (1 - e_out) *
                  ((1 + _q_out) * (1 + e_out) / math.sqrt(1 - e_out))**0.4)

print(f"\n── Mardling stability ───────────────────────────────")
print(f"  a_out/a_in actual   = {ratio_actual:.1f}")
print(f"  a_out/a_in required = {ratio_mardling:.2f}")
print(f"  {'[STABLE]' if ratio_actual > ratio_mardling else '[UNSTABLE]'}")

cos_i    = math.cos(math.radians(i_deg))
e_max_KL_analytic = math.sqrt(max(0.0, 1.0 - 5/3 * cos_i**2))
print(f"\n── Analytic KL prediction ───────────────────────────")
print(f"  e_max (quadrupole) = {e_max_KL_analytic:.4f}")
print(f"  (actual REBOUND result may differ slightly due to EKL)")

if _HAVE_REBOUNDX:
    dperi = 6 * math.pi * G * M_in / (c**2 * a_in * (1 - e_in**2))
    P_in  = 2 * math.pi * math.sqrt(a_in**3 / (G * M_in))
    print(f"\n── 1PN periastron advance (inner binary) ────────────")
    print(f"  Delta_omega/orbit = {math.degrees(dperi):.4e} deg")
    print(f"  Precession period = {(2*math.pi/(dperi/P_in))/yr:.3e} yr")
    print(f"  T_KL/T_precess    = "
          f"{T_KL(m1,m2,m3,a_in,a_out,e_out)/(2*math.pi/(dperi/P_in)):.4f}  "
          f"(≈ epsilon_GR at 1 AU)")

In [ ]:
# KL evolution: direct N-body integration of the fiducial triple over ~10 KL cycles to measure e_min, e_max and plot e(t).

import time as time_module

sim, rebx = build_triple(m1, m2, m3, a_in, e_in,
                   a_out, e_out, i_deg)

N_KL_cycles = 10
T_KL_est    = T_KL(m1, m2, m3, a_in, a_out, e_out)
t_end       = N_KL_cycles * T_KL_est
N_steps     = 500_000
dt_out      = t_end / N_steps

print(f"Triple evolution parameters:")
print(f"  T_KL estimate = {T_KL_est/yr:.2f} yr")
print(f"  Duration      = {t_end/yr:.1f} yr "
      f"({N_KL_cycles} KL cycles)")
print(f"  N_steps       = {N_steps}")
print(f"  dt_out        = {dt_out/yr:.4f} yr")
print(f"\nRunning...")

rec_t   = []
rec_ain = []
rec_ein = []
rec_i   = []

t_now = 0.0
t0    = time_module.time()

for step in range(N_steps):
    sim.integrate(t_now + dt_out, exact_finish_time=1)
    t_now += dt_out

    p0 = sim.particles[0]
    p1 = sim.particles[1]
    p2 = sim.particles[2]

    orb_in = p1.orbit(primary=p0)
    a_in_now = float(np.clip(orb_in.a,  1e3, 1e15))
    e_in_now = float(np.clip(orb_in.e,  0.0, 0.9999))

    dx_in  = p1.x  - p0.x
    dy_in  = p1.y  - p0.y
    dz_in  = p1.z  - p0.z
    dvx_in = p1.vx - p0.vx
    dvy_in = p1.vy - p0.vy
    dvz_in = p1.vz - p0.vz
    Lx_in  = dy_in*dvz_in  - dz_in*dvy_in
    Ly_in  = dz_in*dvx_in  - dx_in*dvz_in
    Lz_in  = dx_in*dvy_in  - dy_in*dvx_in
    Lmag_in = math.sqrt(Lx_in**2 + Ly_in**2 + Lz_in**2) + 1e-30

    m01 = p0.m + p1.m
    cx  = (p0.m*p0.x  + p1.m*p1.x)  / m01
    cy  = (p0.m*p0.y  + p1.m*p1.y)  / m01
    cz  = (p0.m*p0.z  + p1.m*p1.z)  / m01
    cvx = (p0.m*p0.vx + p1.m*p1.vx) / m01
    cvy = (p0.m*p0.vy + p1.m*p1.vy) / m01
    cvz = (p0.m*p0.vz + p1.m*p1.vz) / m01

    dx_out  = p2.x  - cx
    dy_out  = p2.y  - cy
    dz_out  = p2.z  - cz
    dvx_out = p2.vx - cvx
    dvy_out = p2.vy - cvy
    dvz_out = p2.vz - cvz
    Lx_out  = dy_out*dvz_out - dz_out*dvy_out
    Ly_out  = dz_out*dvx_out - dx_out*dvz_out
    Lz_out  = dx_out*dvy_out - dy_out*dvx_out
    Lmag_out = math.sqrt(Lx_out**2 + Ly_out**2 + Lz_out**2) + 1e-30

    cos_i_now = ((Lx_in*Lx_out + Ly_in*Ly_out + Lz_in*Lz_out)
                 / (Lmag_in * Lmag_out))
    i_now = math.degrees(
        math.acos(max(-1.0, min(1.0, cos_i_now))))

    rec_t.append(t_now)
    rec_ain.append(a_in_now)
    rec_ein.append(e_in_now)
    rec_i.append(i_now)

    if step % 50000 == 0:
        elapsed = time_module.time() - t0
        print(f"  step {step}/{N_steps}  "
              f"t={t_now/yr:.1f} yr  "
              f"a={a_in_now/AU:.4f} AU  "
              f"e={e_in_now:.4f}  "
              f"i={i_now:.2f} deg  "
              f"({elapsed:.0f}s elapsed)")

rec_t   = np.array(rec_t)
rec_ain = np.array(rec_ain)
rec_ein = np.array(rec_ein)
rec_i   = np.array(rec_i)

np.save('triple_t.npy',   rec_t)
np.save('triple_ain.npy', rec_ain)
np.save('triple_ein.npy', rec_ein)
np.save('triple_i.npy',   rec_i)

e_min_KL = rec_ein.min()
e_max_KL = rec_ein.max()
i_min_KL = rec_i.min()
i_max_KL = rec_i.max()

a_hardened = a_decouple(e_max_KL)
print(f"a_hardened = {a_hardened/AU:.4f} AU  (decoupling; e_ret = {e_max_KL:.3f})")

print(f"\nTriple evolution complete.")
print(f"  e_in range : {e_min_KL:.4f} → {e_max_KL:.4f}")
print(f"  i range    : {i_min_KL:.2f} → {i_max_KL:.2f} deg")
print(f"  a_in range : {rec_ain.min()/AU:.6f} → "
      f"{rec_ain.max()/AU:.6f} AU  "
      f"(should be ~constant)")
print(f"  Analytic e_max prediction: {e_max_KL_analytic:.4f}")
print(f"  Agreement: "
      f"{'GOOD' if abs(e_max_KL - e_max_KL_analytic) < 0.1 else 'CHECK — EKL may be significant'}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6),
                                sharex=True,
                                gridspec_kw={'hspace': 0.08})

ax1.plot(rec_t/yr, rec_ein, color=C_TRIPLE, lw=0.7)
ax1.axhline(e_min_KL, color='#888', lw=0.8, ls='--', alpha=0.6,
            label=f'$e_{{\\rm min}}={e_min_KL:.3f}$')
ax1.axhline(e_max_KL, color=C_TRIPLE, lw=0.8, ls='--', alpha=0.6,
            label=f'$e_{{\\rm max}}={e_max_KL:.3f}$ '
                  f'(analytic: {e_max_KL_analytic:.3f})')
ax1.axhline(e_max_KL_analytic, color=C_DECOUPLE,
            lw=0.8, ls=':', alpha=0.8,
            label='Analytic prediction')
ax1.set_ylabel(r'Inner eccentricity $e_{\rm in}$', labelpad=6)
ax1.set_title(f'KL validation — test parameters '
              f'($m_1=m_2={m1/M_sun:.0f}$ $M_\\odot$, '
              f'$a_{{\\rm out}}/a_{{\\rm in}}=30$, '
              f'$i_0={i_deg}°$)',
              loc='left', fontsize=11)
ax1.legend(fontsize=9, frameon=False)
style_ax(ax1)

ax2.plot(rec_t/yr, rec_i, color=C_ISO, lw=0.7)
ax2.set_xlabel('Time (yr)', labelpad=6)
ax2.set_ylabel(r'Mutual inclination $i$ (deg)', labelpad=6)
style_ax(ax2)

plt.show()
print("\nCell 4 complete.")
print("GO/NO-GO:")
print(f"  KL oscillating?     "
      f"{'YES' if e_max_KL - e_min_KL > 0.2 else 'NO — check parameters'}")
print(f"  Inclination correct? "
      f"{'YES' if abs(i_max_KL - i_deg) < 5 else 'NO — check Cell 3 build_triple'}")
print(f"  a_in stable?        "
      f"{'YES' if (rec_ain.max()-rec_ain.min())/AU < 0.01 else 'NO — GW effects or instability'}")

In [ ]:
# Peters inspiral: two-stage orbit-averaged inspiral from the handoff separation to ISCO for the KL-max, KL-min and circular cases.

from scipy.integrate import solve_ivp

a_ISCO   = isco_radius(m1, m2)
a_10Hz   = a_from_frequency(f_band_entry, m1, m2)
F_SEAM   = 3.0
a_seam   = a_from_frequency(F_SEAM, m1, m2)
STAGEB_MAX_STEP = 0.25

print("Fixed-span two-stage Peters setup:")
print(f"  a(10 Hz)   = {a_10Hz:.4e} m = {a_10Hz/AU:.8f} AU")
print(f"  f_seam     = {F_SEAM:.1f} Hz  ->  a_seam = {a_seam:.4e} m "
      f"({a_seam/a_10Hz:.2f} x a(10Hz))")
print(f"  a_ISCO     = {a_ISCO:.4e} m")

def _band_index(f_array, f_entry):
    """Index of band entry, plus whether the band was actually reached."""
    idx = int(np.searchsorted(f_array, f_entry))
    if idx >= len(f_array):
        return len(f_array) - 1, False
    return idx, True

def run_peters(a0, e0, label="", n_coarse=100000,
               rk4_target_seconds=5.0, m1=None, m2=None):
    """
    Fixed-span two-stage Peters orbit-averaged inspiral.

    Stage A: a0     -> a_seam  (coarse)  — pre-band inspiral, cheap
    Stage B: a_seam -> ISCO    (fine)    — the LIGO band, densely sampled

    The Stage B span is set by the fixed frequency f_seam, NOT by a0, so the
    number of in-band samples is independent of the handoff separation. If the
    handoff already sits below a_seam (deep handoff, e.g. a sweep point), Stage A
    is skipped and the whole thing is the fine stage.

    Returns a dict with:
        t, t_fine, t_merge, a, e, f_gw,
        idx_band, reached_band, e_at_band, f_at_band,
        a_rk4, e_rk4, t_rk4_left, label
    where 't' is absolute time (coarse near merger; plots/comparisons only) and
    't_fine' is a precise merger-relative clock (use this for the waveform).
    e_at_band is NaN when reached_band is False.
    """
    if m1 is None:
        m1 = globals()["m1"]
    if m2 is None:
        m2 = globals()["m2"]

    M_in   = m1 + m2
    a_ISCO = isco_radius(m1, m2)
    a_seam = a_from_frequency(F_SEAM, m1, m2)

    def _make_peters_rhs(m1, m2, M_in):
        def peters_rhs(t, y):
            a, e = y
            e = float(np.clip(e, 1e-9, 0.9999))
            beta = 64/5 * G**3 * m1 * m2 * M_in / c**5
            f_e = (1 + 73/24*e**2 + 37/96*e**4) / (1 - e**2)**3.5
            g_e = (1 + 121/304*e**2) / (1 - e**2)**2.5
            da_dt = -beta * f_e / a**3
            de_dt = -(304/15) * e * G**3 * m1 * m2 * M_in / (c**5 * a**4) * g_e
            return [da_dt, de_dt]
        return peters_rhs

    peters_rhs = _make_peters_rhs(m1, m2, M_in)
    T_hubble = 13.8e9 * yr

    def event_seam(t, y):
        return y[0] - a_seam
    event_seam.terminal = True
    event_seam.direction = -1

    def event_ISCO(t, y):
        return y[0] - 1.01 * a_ISCO
    event_ISCO.terminal = True
    event_ISCO.direction = -1

    if a0 > a_seam:
        sol_A = solve_ivp(
            peters_rhs, [0.0, T_hubble], [a0, e0],
            method='RK45', events=[event_seam, event_ISCO],
            rtol=1e-9, atol=1e-11, max_step=T_hubble / n_coarse,
        )
        a_A = np.asarray(sol_A.y[0], dtype=float)
        e_A = np.clip(np.asarray(sol_A.y[1], dtype=float), 0.0, 0.9999)
        t_A = np.asarray(sol_A.t, dtype=float)

        if a_A[-1] > 2.0 * a_seam:
            print(f"  [{label}] WARNING: Stage A never reached the seam "
                  f"(final a = {a_A[-1]:.3e} m); binary may not merge "
                  f"within a Hubble time.")
            f_A = np.array([gw_frequency(a, m1, m2) for a in a_A])
            idx_band, reached = _band_index(f_A, f_band_entry)
            return {
                "t": t_A, "t_fine": t_A - float(t_A[-1]), "t_merge": float(t_A[-1]),
                "a": a_A, "e": e_A, "f_gw": f_A,
                "idx_band": idx_band, "reached_band": reached,
                "e_at_band": float(e_A[idx_band]) if reached else float('nan'),
                "f_at_band": float(f_A[idx_band]) if reached else float('nan'),
                "a_rk4": float(a_A[-1]), "e_rk4": float(e_A[-1]),
                "t_rk4_left": np.nan, "label": label,
            }
        a_B0, e_B0, t_B0 = float(a_A[-1]), float(e_A[-1]), float(t_A[-1])
    else:

        a_A = np.array([a0]); e_A = np.array([e0]); t_A = np.array([0.0])
        a_B0, e_B0, t_B0 = float(a0), float(e0), 0.0

    print(f"  [{label}] seam: a={a_B0:.4e} m  e={e_B0:.6f}  "
          f"f={gw_frequency(a_B0, m1, m2):.3f} Hz  (t={t_B0/yr:.3e} yr)")

    T_B_window = 1.0e6 * yr
    sol_B = solve_ivp(
        peters_rhs, [0.0, T_B_window], [a_B0, e_B0],
        method='RK45', events=[event_ISCO],
        rtol=1e-10, atol=1e-12, max_step=STAGEB_MAX_STEP,
    )
    a_B    = np.asarray(sol_B.y[0], dtype=float)
    e_B    = np.clip(np.asarray(sol_B.y[1], dtype=float), 0.0, 0.9999)
    tB_loc = np.asarray(sol_B.t, dtype=float)

    _keep = a_B >= 1.0 * a_ISCO
    if not np.all(_keep):
        a_B, e_B, tB_loc = a_B[_keep], e_B[_keep], tB_loc[_keep]
    if len(a_B) < 2:
        a_B = np.array([a_B0]); e_B = np.array([e_B0]); tB_loc = np.array([0.0])

    t_merge_abs = t_B0 + float(tB_loc[-1])
    tB_abs  = tB_loc + t_B0
    tB_fine = tB_loc - float(tB_loc[-1])

    print(f"  [{label}] Stage B: {len(a_B)} samples  "
          f"end a={a_B[-1]:.4e} m ({a_B[-1]/a_ISCO:.3f} x a_ISCO)  "
          f"f={gw_frequency(a_B[-1], m1, m2):.2f} Hz")

    if a0 > a_seam and len(a_B) > 1:
        a_arr  = np.concatenate([a_A, a_B[1:]])
        e_arr  = np.concatenate([e_A, e_B[1:]])
        t_arr  = np.concatenate([t_A, tB_abs[1:]])
        t_fine = np.concatenate([t_A - t_merge_abs, tB_fine[1:]])
    else:
        a_arr, e_arr = a_B.copy(), e_B.copy()
        t_arr, t_fine = tB_abs.copy(), tB_fine.copy()
    f_arr = np.array([gw_frequency(a, m1, m2) for a in a_arr])

    idx_band, reached = _band_index(f_arr, f_band_entry)
    n_after = len(f_arr) - idx_band
    if not reached:
        print(f"    WARNING [{label}]: band NOT reached; "
              f"max f = {f_arr[-1]:.4f} Hz < {f_band_entry} Hz -> e_at_band=NaN")
    elif n_after < 16:
        print(f"    WARNING [{label}]: only {n_after} samples after band entry "
              f"(waveform builder may struggle).")

    t_to_merge_arr = -t_fine
    below_target = np.where(t_to_merge_arr <= rk4_target_seconds)[0]
    idx_rk4 = int(below_target[0]) if len(below_target) > 0 else len(a_arr) - 1

    _e_band = float(e_arr[idx_band]) if reached else float('nan')
    _f_band = float(f_arr[idx_band]) if reached else float('nan')

    print(f"  [{label}] Summary: t_merge={t_merge_abs/yr:.4e} yr  "
          f"e(10Hz)={_e_band:.6e}  post-band samples={n_after}")

    return {
        "t": t_arr, "t_fine": t_fine, "t_merge": t_merge_abs,
        "a": a_arr, "e": e_arr, "f_gw": f_arr,
        "idx_band": idx_band, "reached_band": reached,
        "e_at_band": _e_band, "f_at_band": _f_band,
        "a_rk4": float(a_arr[idx_rk4]), "e_rk4": float(e_arr[idx_rk4]),
        "t_rk4_left": float(t_to_merge_arr[idx_rk4]), "label": label,
    }

print("\nRunning fixed-span Peters inspirals (per-case decoupling handoff)...\n")

res_KL_max = run_peters(a_decouple(e_max_KL), e_max_KL, label="KL max")
print()
res_KL_min = run_peters(a_decouple(e_min_KL), e_min_KL, label="KL min")
print()
res_circ   = run_peters(a_decouple(0.001),    0.001,    label="Isolated circular")
print()
res_field  = run_peters(a_decouple(e_in),     e_in,     label=f"Field binary (e={e_in})")

print(f"\n{'='*54}")
print(f"Residual eccentricity at {f_band_entry} Hz")
print(f"{'='*54}")
for _nm, _r in [("KL max", res_KL_max), ("KL min", res_KL_min),
                ("Field", res_field), ("Circular", res_circ)]:
    _flag = "" if _r["reached_band"] else "   [BAND NOT REACHED]"
    print(f"  {_nm:9s}: e(10Hz) = {_r['e_at_band']:.6e}{_flag}")

print(f"\n  KL max / circular : "
      f"{res_KL_max['e_at_band']/max(res_circ['e_at_band'],1e-30):.1f}x")
print(f"  KL range          : "
      f"{res_KL_min['e_at_band']:.6e} → {res_KL_max['e_at_band']:.6e}")

print("\nCell 6 complete.")

In [ ]:
# Residual eccentricity: e(t) and f_GW(t), merger-time shortening from KL, and e at 10 Hz across the KL eccentricity range.

e_KL_range      = np.linspace(rec_ein.min(), rec_ein.max(), 30)
e_at_band_study = np.full(len(e_KL_range), np.nan)
t_merge_study   = np.full(len(e_KL_range), np.nan)

print("Running parameter study over KL eccentricity range "
      "(per-case decoupling handoff)...")
for i, e_i in enumerate(e_KL_range):
    e_i = float(e_i)
    res_i = run_peters(a_decouple(e_i), e_i, label="", n_coarse=50000)
    e_at_band_study[i] = res_i['e_at_band']
    t_merge_study[i]   = res_i['t'][-1]

t_merge_KL    = res_KL_max['t'][-1]
t_merge_circ  = res_circ['t'][-1]
t_merge_field = res_field['t'][-1]

print(f"\n── Merger time comparison ───────────────────────────")
print(f"  KL max       : {t_merge_KL/yr:.4e} yr")
print(f"  Circular ref : {t_merge_circ/yr:.4e} yr")
print(f"  Field binary : {t_merge_field/yr:.4e} yr")
print(f"  KL speedup vs circular : {t_merge_circ/t_merge_KL:.2f}x faster")
print(f"  KL speedup vs field    : {t_merge_field/t_merge_KL:.2f}x faster")

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.30,
                        left=0.08, right=0.97,
                        top=0.93, bottom=0.08)

ax_e    = fig.add_subplot(gs[0, 0])
ax_f    = fig.add_subplot(gs[0, 1])
ax_study= fig.add_subplot(gs[1, 0])
ax_acc  = fig.add_subplot(gs[1, 1])

for res, col, ls, lab in [
        (res_KL_max, C_TRIPLE, '-',  'KL max'),
        (res_KL_min, C_WAVE,   '--', 'KL min'),
        (res_circ,   C_ISO,    '--', 'Circular ref'),
        (res_field,  C_DECOUPLE,':','Field binary')]:
    ax_e.plot(res['t']/yr, res['e'],
              color=col, lw=1.1, ls=ls, label=lab)
    if res['idx_band'] < len(res['e']):
        ax_e.scatter(res['t'][res['idx_band']]/yr,
                     res['e_at_band'],
                     color=col, s=50, zorder=5)

ax_e.set_xlabel('Time (yr)', labelpad=5)
ax_e.set_ylabel(r'Eccentricity $e$', labelpad=5)
ax_e.set_title('(a) Eccentricity evolution',
               loc='left', fontsize=11, pad=7)
ax_e.legend(fontsize=8, frameon=False)
style_ax(ax_e)

for res, col, ls, lab in [
        (res_KL_max, C_TRIPLE, '-',  'KL max'),
        (res_KL_min, C_WAVE,   '--', 'KL min'),
        (res_circ,   C_ISO,    '--', 'Circular ref'),
        (res_field,  C_DECOUPLE,':','Field binary')]:
    ax_f.semilogy(res['t']/yr, res['f_gw'],
                  color=col, lw=1.1, ls=ls, label=lab)
ax_f.axhline(f_band_entry, color='#444',
             lw=1.0, ls=':', label=f'{f_band_entry} Hz')
ax_f.set_xlabel('Time (yr)', labelpad=5)
ax_f.set_ylabel(r'$f_{\rm gw}$ (Hz)', labelpad=5)
ax_f.set_title('(b) GW frequency chirp',
               loc='left', fontsize=11, pad=7)
ax_f.legend(fontsize=8, frameon=False)
style_ax(ax_f)

E_DETECTABLE = 1.0e-2
ax_study.semilogy(e_KL_range, e_at_band_study,
                  color=C_TRIPLE, lw=0.9, marker='o', ms=4, zorder=4,
                  label=r'Residual $e$(10 Hz)')
ax_study.axhline(res_circ['e_at_band'],
                 color=C_ISO, lw=1.2, ls='--',
                 label='Circular reference')
ax_study.axhline(E_DETECTABLE, color='#444', lw=1.2, ls=':',
                 label=r'$e$ for $\mathcal{M}=0.03$ threshold')
ax_study.axvline(e_max_KL, color=C_DECOUPLE,
                 lw=1.0, ls='--', alpha=0.7,
                 label=f'$e_{{\\rm max}}={e_max_KL:.3f}$')
ax_study.set_xlabel(r'$e$ retained at decoupling', labelpad=5)
ax_study.set_ylabel(r'Residual $e$ at $f_{\rm gw}=10\,{\rm Hz}$', labelpad=5)
ax_study.set_title(r'(c) KL-phase sensitivity — primary result (null)',
                   loc='left', fontsize=11, pad=7)
ax_study.legend(fontsize=8, frameon=False)
style_ax(ax_study)

ax_acc.semilogy(e_KL_range, t_merge_study/yr,
                color=C_TRIPLE, lw=1.2)
ax_acc.axhline(t_merge_circ/yr, color=C_ISO,
               lw=1.2, ls='--',
               label='Circular reference')
ax_acc.axhline(t_merge_field/yr, color=C_DECOUPLE,
               lw=1.0, ls=':', alpha=0.8,
               label='Field binary')
ax_acc.axvline(e_max_KL, color='#888',
               lw=0.8, ls='--', alpha=0.6)
ax_acc.set_xlabel(r'$e$ retained at decoupling', labelpad=5)
ax_acc.set_ylabel('Merger time (yr)', labelpad=5)
ax_acc.set_title('(d) Merger time acceleration',
                 loc='left', fontsize=11, pad=7)
ax_acc.legend(fontsize=8, frameon=False)
style_ax(ax_acc)

plt.show()

print("\nCell 7 complete.")
print(f"  e(10Hz) range across KL cycle: "
      f"{np.nanmin(e_at_band_study):.3e} → {np.nanmax(e_at_band_study):.3e}")
print(f"  Circular reference           : {res_circ['e_at_band']:.3e}")
print(f"  Detection-threshold e (~M=0.03): {E_DETECTABLE:.1e}  "
      f"-> residual is ~{E_DETECTABLE/np.nanmax(e_at_band_study):.0e}x below it")

In [ ]:
# Waveforms: Peters-driven eccentric harmonic waveforms for the KL-max, KL-min and circular cases.

from scipy.interpolate import PchipInterpolator
from scipy.integrate import cumulative_trapezoid, solve_ivp
from scipy.special import jv

def peters_g_n(n, e):
    e = np.asarray(e, dtype=float)
    ne = n * e

    Jm2 = jv(n - 2, ne)
    Jm1 = jv(n - 1, ne)
    J0  = jv(n,     ne)
    Jp1 = jv(n + 1, ne)
    Jp2 = jv(n + 2, ne)

    A = Jm2 - 2.0*e*Jm1 + (2.0/n)*J0 + 2.0*e*Jp1 - Jp2
    B = Jm2 - 2.0*J0 + Jp2

    g = (n**4 / 32.0) * (
        A**2
        + (1.0 - e**2) * B**2
        + (4.0 / (3.0 * n**2)) * J0**2
    )
    return np.maximum(g, 0.0)

def gw_peak_frequency(a, e, m1, m2):
    M = m1 + m2
    a = np.asarray(a, dtype=float)
    e = np.asarray(e, dtype=float)

    return (
        np.sqrt(G * M) / np.pi
        * (1.0 + e)**1.1954
        / (a * (1.0 - e**2))**1.5
    )

def generate_harmonic_waveform(res, label="", n_max=15, iota_deg=0.0,
                               samples_per_highest_cycle=24, fgw_cap=f_gw_max,
                               m1=None, m2=None):
    """
    Build h_+(t), h_x(t) from Peters inspiral using harmonic synthesis.
    Starts at 10 Hz band entry and ends at a physically capped GW frequency.

    Masses default to the module globals m1, m2 when not supplied, so
    existing no-mass calls are unchanged. When masses ARE passed, the
    mass-dependent separation a_ISCO is recomputed locally from them.
    """

    if m1 is None:
        m1 = globals()["m1"]
    if m2 is None:
        m2 = globals()["m2"]

    a_ISCO = isco_radius(m1, m2)

    idx0 = int(res['idx_band'])
    t_src = np.asarray(res.get('t_fine', res['t'])[idx0:], dtype=float)
    a_src = np.asarray(res['a'][idx0:], dtype=float)
    e_src = np.asarray(res['e'][idx0:], dtype=float)

    if len(t_src) < 5:
        raise RuntimeError(f"[{label}] Not enough Peters samples after band entry.")

    t_src = t_src - t_src[0]

    good = np.isfinite(t_src) & np.isfinite(a_src) & np.isfinite(e_src)
    t_src = t_src[good]
    a_src = a_src[good]
    e_src = e_src[good]

    keep = np.concatenate(([True], np.diff(t_src) > 0.0))
    t_src = t_src[keep]
    a_src = a_src[keep]
    e_src = e_src[keep]

    if len(t_src) < 5:
        raise RuntimeError(f"[{label}] Not enough clean Peters samples after cleanup.")

    M = m1 + m2
    eta_gw = m1 * m2 / M**2
    iota = math.radians(iota_deg)

    a_end = 1.01 * a_ISCO
    if a_src[-1] > a_end:
        def _peters_rhs(_t, y):
            a, e = y
            e = min(max(e, 0.0), 0.999999)
            one_me2 = max(1.0 - e * e, 1e-12)
            pref = G**3 * m1 * m2 * M / c**5
            da = -(64.0 / 5.0) * pref / a**3 * \
                 (1.0 + 73.0/24.0*e**2 + 37.0/96.0*e**4) / one_me2**3.5
            de = -(304.0/15.0) * e * pref / a**4 * \
                 (1.0 + 121.0/304.0*e**2) / one_me2**2.5
            return [da, de]

        def _hit_isco(_t, y):
            return y[0] - a_end
        _hit_isco.terminal = True
        _hit_isco.direction = -1

        T_rem = (5.0/256.0) * c**5 * a_src[-1]**4 / (G**3 * m1 * m2 * M) \
                * (1.0 - e_src[-1]**2)**3.5
        _sol = solve_ivp(_peters_rhs,
                         [t_src[-1], t_src[-1] + 3.0*T_rem + 1.0],
                         [a_src[-1], e_src[-1]], method="RK45",
                         events=_hit_isco, rtol=1e-10, atol=1e-12,
                         max_step=max(T_rem/4000.0, 1e-6))
        if _sol.t.size > 1 and _sol.y[0][-1] <= 1.05 * a_end:
            t_src = np.concatenate([t_src, _sol.t[1:]])
            a_src = np.concatenate([a_src, np.clip(_sol.y[0][1:], a_end, None)])
            e_src = np.concatenate([e_src, np.clip(_sol.y[1][1:], 0.0, 0.9999)])

    a_of_t = PchipInterpolator(t_src, a_src, extrapolate=False)
    e_of_t = PchipInterpolator(t_src, e_src, extrapolate=False)

    f_orb_src = 0.5 * np.array([gw_frequency(a, m1, m2) for a in a_src])
    f_hi_src = n_max * f_orb_src
    f_hi_max = max(float(np.max(f_hi_src)),
                   n_max * 0.5 * float(gw_frequency(a_end, m1, m2)),
                   2.0)

    dt = 1.0 / (samples_per_highest_cycle * f_hi_max)
    dt = min(dt, 1.0 / 4096.0)
    dt = max(dt, 1.0e-5)

    t_end = float(t_src[-1])
    t_uni = np.arange(0.0, t_end + dt, dt)
    if len(t_uni) < 16:
        raise RuntimeError(f"[{label}] Uniform waveform grid too short.")

    a_uni = np.asarray(a_of_t(t_uni), dtype=float)
    e_uni = np.asarray(e_of_t(t_uni), dtype=float)

    finite = np.isfinite(a_uni) & np.isfinite(e_uni)
    t_uni = t_uni[finite]
    a_uni = a_uni[finite]
    e_uni = e_uni[finite]
    if len(t_uni) < 16:
        raise RuntimeError(f"[{label}] Too few finite interpolated samples.")

    a_min = max(1.01 * a_ISCO, np.min(a_src))
    a_max = np.max(a_src)
    a_uni = np.clip(a_uni, a_min, a_max)
    e_uni = np.clip(e_uni, 0.0, 0.9999)

    fgw_uni = gw_peak_frequency(a_uni, e_uni, m1, m2)

    if np.isfinite(fgw_cap):
        keep_band = np.isfinite(fgw_uni) & (fgw_uni <= fgw_cap)
        if np.any(keep_band):
            last_good = np.where(keep_band)[0][-1] + 1
            t_uni = t_uni[:last_good]
            a_uni = a_uni[:last_good]
            e_uni = e_uni[:last_good]
            fgw_uni = fgw_uni[:last_good]

    if len(t_uni) < 16:
        raise RuntimeError(f"[{label}] Waveform fully trimmed by frequency cap.")

    f_isco_target = float(gw_peak_frequency(a_end, 0.0, m1, m2))
    reached_isco = bool(fgw_uni[-1] >= 0.90 * f_isco_target)
    if not reached_isco:
        print(f"[{label}] WARNING: terminal f_peak {fgw_uni[-1]:.1f} Hz < "
              f"{0.90*f_isco_target:.1f} Hz (0.9 x ISCO {f_isco_target:.1f} Hz) "
              f"— waveform truncated; mismatch against it is unreliable.")

    n_orb_uni = np.sqrt(G * M / a_uni**3)
    ell = cumulative_trapezoid(n_orb_uni, t_uni, initial=0.0)

    x_uni = G * M / (a_uni * c**2)
    A0 = 4.0 * G * M * eta_gw * x_uni / (c**2 * R)

    hp = np.zeros_like(t_uni)
    hx = np.zeros_like(t_uni)

    cos_iota = math.cos(iota)
    Fp = 0.5 * (1.0 + cos_iota**2)
    Fx = cos_iota

    for n in range(1, n_max + 1):
        g_n = peters_g_n(n, e_uni)
        amp_n = A0 * np.sqrt(g_n) / n
        phase_n = n * ell
        hp += Fp * amp_n * np.cos(phase_n)
        hx += Fx * amp_n * np.sin(phase_n)

    hp = np.nan_to_num(hp, nan=0.0, posinf=0.0, neginf=0.0)
    hx = np.nan_to_num(hx, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"[{label}] harmonic waveform generated")
    print(f"    start at band entry  : f_peak ~ {fgw_uni[0]:.3f} Hz")
    print(f"    end near merger      : f_peak ~ {fgw_uni[-1]:.3f} Hz")
    print(f"    reached ISCO         : {reached_isco}")
    print(f"    duration             : {t_uni[-1]:.4f} s")
    print(f"    dt                   : {dt:.6e} s")
    print(f"    samples              : {len(t_uni)}")
    print(f"    n_max                : {n_max}")
    print(f"    e range in waveform  : {e_uni[0]:.6f} -> {e_uni[-1]:.6f}")
    print(f"    a range in waveform  : {a_uni[0]:.6e} -> {a_uni[-1]:.6e} m")
    print(f"    max |h_plus|         : {np.max(np.abs(hp)):.3e}")
    print(f"    max |h_cross|        : {np.max(np.abs(hx)):.3e}")
    print()

    return {
        "ts": t_uni,
        "hps": hp,
        "hxs": hx,
        "fgw": fgw_uni,
        "a": a_uni,
        "e": e_uni,
        "mean_anomaly": ell,
        "reached_isco": reached_isco,
        "label": label,
    }

print("Generating Peters-driven harmonic waveforms...")
print("=" * 58)

wf_KL = generate_harmonic_waveform(
    res_KL_max,
    label="KL-driven",
    n_max=15,
    iota_deg=0.0
)

wf_circ = generate_harmonic_waveform(
    res_circ,
    label="Circular",
    n_max=15,
    iota_deg=0.0
)

wf_field = generate_harmonic_waveform(
    res_field,
    label="Field binary",
    n_max=15,
    iota_deg=0.0
)

print("=" * 58)
print("Waveform summary:")
for name, wf in [("KL-driven", wf_KL),
                 ("Circular",  wf_circ),
                 ("Field",     wf_field)]:
    print(f"  [{name}]  "
          f"dur={wf['ts'][-1]-wf['ts'][0]:.4f} s  "
          f"f_peak: {wf['fgw'][0]:.2f} -> {wf['fgw'][-1]:.2f} Hz  "
          f"h_max={np.max(np.abs(wf['hps'])):.3e}")

print("\nCell 8 complete.")
print("Next: rerun waveform plotting cell, then Cell 8b.")

In [ ]:
# Faithfulness: PSD-weighted, time- and phase-maximized overlap and mismatch between KL and circular waveforms versus KL phase.

import time as time_mod
import numpy as np

C_KL_shade   = '#8c1d18'
C_NULL_shade = '#1a5c38'

DT_RESAMPLE = 1e-5
F_LOW       = 10.0
F_HIGH      = 1024.0
TAU_MAX     = 0.1

def faithfulness_fd(wf_signal, wf_template,
                    f_low=F_LOW, f_high=F_HIGH,
                    dt_resample=DT_RESAMPLE, tau_max=TAU_MAX, debug=False):
    """
    Faithfulness against the aLIGO ZeroDetHighPower PSD.

        <a|b> = 4 Re int_{f_low}^{f_high} a*(f) b(f) / S_n(f) df
        F = max_{t_c, phi_c} |<h1|h2>| / sqrt(<h1|h1> <h2|h2>)

    Waveforms aligned at MERGER; time-shift maximization restricted to
    |t_c| <= tau_max; complex inverse transform does the constant-phase
    maximization. Faithfulness WITHIN our own waveform family — necessary but not
    sufficient for detectability by a real search (needs SEOBNRE/TEOBResumS).
    """
    if not _HAVE_PYCBC:
        raise RuntimeError("pycbc is required for faithfulness_fd. "
                           "Install: pip install pycbc")

    ts1 = np.asarray(wf_signal['ts'],   dtype=float)
    h1  = np.asarray(wf_signal['hps'],  dtype=float)
    ts2 = np.asarray(wf_template['ts'], dtype=float)
    h2  = np.asarray(wf_template['hps'],dtype=float)

    ok1 = np.isfinite(ts1) & np.isfinite(h1)
    ok2 = np.isfinite(ts2) & np.isfinite(h2)
    ts1, h1 = ts1[ok1], h1[ok1]
    ts2, h2 = ts2[ok2], h2[ok2]
    if len(ts1) < 16 or len(ts2) < 16:
        if debug: print("    [fd] waveform too short")
        return np.nan, np.nan

    fg1 = np.asarray(wf_signal.get('fgw',   []), dtype=float)
    fg2 = np.asarray(wf_template.get('fgw', []), dtype=float)
    if fg1.size and fg2.size:
        fg1, fg2 = fg1[ok1], fg2[ok2]
        if fg1.size == len(ts1) and fg2.size == len(ts2):
            f_end = min(float(fg1[-1]), float(fg2[-1]))
            k1 = np.where(fg1 <= f_end)[0]
            k2 = np.where(fg2 <= f_end)[0]
            if len(k1) >= 16 and len(k2) >= 16:
                ts1, h1 = ts1[:k1[-1] + 1], h1[:k1[-1] + 1]
                ts2, h2 = ts2[:k2[-1] + 1], h2[:k2[-1] + 1]
                if debug:
                    print(f"    [fd] terminal anchor: f_end={f_end:.1f} Hz  "
                          f"len1={len(ts1)} len2={len(ts2)}")

    ts1 = ts1 - ts1[-1]
    ts2 = ts2 - ts2[-1]
    T_common = min(abs(float(ts1[0])), abs(float(ts2[0])))
    if T_common <= 0:
        if debug: print("    [fd] non-positive common duration")
        return np.nan, np.nan

    dt     = max(float(dt_resample), 1e-6)
    t_grid = np.arange(-T_common, 0.0, dt)
    if len(t_grid) < 32:
        if debug: print("    [fd] resampled grid too short")
        return np.nan, np.nan

    h1r = np.interp(t_grid, ts1, h1, left=0.0, right=0.0)
    h2r = np.interp(t_grid, ts2, h2, left=0.0, right=0.0)
    h1r -= np.mean(h1r)
    h2r -= np.mean(h2r)

    N  = int(2 ** np.ceil(np.log2(len(t_grid)))) * 2
    p1 = np.zeros(N); p2 = np.zeros(N)
    p1[:len(h1r)] = h1r
    p2[:len(h2r)] = h2r

    H1    = np.fft.fft(p1)
    H2    = np.fft.fft(p2)
    freqs = np.fft.fftfreq(N, d=dt)
    df    = 1.0 / (N * dt)

    psd_pos = np.asarray(
        aLIGOZeroDetHighPower(N // 2 + 1, df, max(f_low * 0.5, 1.0)).data)
    psd = np.empty(N)
    psd[:N // 2 + 1] = psd_pos
    psd[N // 2 + 1:] = psd_pos[1:N // 2][::-1]
    psd[psd <= 0] = np.inf

    absf   = np.abs(freqs)
    band   = (absf >= f_low) & (absf <= f_high)
    invpsd = np.zeros(N)
    invpsd[band] = 1.0 / psd[band]
    if band.sum() < 16:
        if debug: print(f"    [fd] only {band.sum()} active bins")
        return np.nan, np.nan

    n1 = 2.0 * df * np.sum(np.abs(H1)**2 * invpsd)
    n2 = 2.0 * df * np.sum(np.abs(H2)**2 * invpsd)
    if not (np.isfinite(n1) and np.isfinite(n2) and n1 > 0 and n2 > 0):
        if debug: print("    [fd] invalid norms")
        return np.nan, np.nan

    integrand = H1 * np.conj(H2) * invpsd
    integrand[N // 2 + 1:] = 0.0
    z = np.fft.ifft(integrand) * N * 2.0 * df * 2.0
    zabs = np.abs(z)

    max_shift = int(round(tau_max / dt))
    max_shift = max(1, min(max_shift, N // 2 - 1))
    mask = np.zeros(N, dtype=bool)
    mask[:max_shift + 1] = True
    mask[N - max_shift:] = True
    zw   = np.where(mask, zabs, 0.0)
    idx  = int(np.argmax(zw))
    peak = float(zw[idx])
    tau  = idx * dt if idx < N // 2 else (idx - N) * dt

    F = float(np.clip(peak / np.sqrt(n1 * n2), 0.0, 1.0))
    M = 1.0 - F

    if debug:
        print(f"    [fd] T={T_common:.2f}s  N={N}  active_bins={band.sum()}  "
              f"n1={n1:.3e}  n2={n2:.3e}  peak={peak:.3e}  "
              f"tau={tau:+.4f}s  F={F:.8f}")

    return F, M

print("── Faithfulness sanity checks (merger-aligned, aLIGO PSD) ──")
print(f"   dt_resample={DT_RESAMPLE:.0e}s  f_low={F_LOW}Hz  "
      f"f_high={F_HIGH}Hz  tau_max={TAU_MAX}s")
print()

checks_passed = True

for name, wf in [("circular", wf_circ),
                 ("KL max",   wf_KL),
                 ("field",    wf_field)]:
    F_s, M_s = faithfulness_fd(wf, wf, f_low=F_LOW, debug=True)
    ok = np.isfinite(F_s) and abs(F_s - 1.0) < 1e-4
    print(f"  {name:10s} vs itself : F={F_s:.8f}  {'[OK]' if ok else '[FAIL]'}")
    if not ok:
        checks_passed = False
    print()

print("  ── near-circular regression check ──")
_res_nc = run_peters(a_decouple(0.002), 0.002, label="", n_coarse=50000)
_wf_nc  = generate_harmonic_waveform(_res_nc, label="nc-check",
                                     n_max=15, iota_deg=0.0)
_F_nc, _M_nc = faithfulness_fd(_wf_nc, wf_circ, debug=True)
nc_ok = np.isfinite(_M_nc) and _M_nc < 0.08
print(f"  near-circular vs circular : M={_M_nc:.4f}  "
      f"{'[OK]' if nc_ok else '[FAIL — alignment regression]'}")
if not nc_ok:
    checks_passed = False
print()

F_kl_c, M_kl_c = faithfulness_fd(wf_KL, wf_circ, debug=True)
print(f"  KL max vs circular : F={F_kl_c:.6f}  M={M_kl_c:.6f}  "
      f"{'[above threshold]' if np.isfinite(M_kl_c) and M_kl_c > 0.03 else '[null — below threshold]'}")
print()

F_field_c, M_field_c = faithfulness_fd(wf_field, wf_circ, debug=True)
print(f"  field vs circular  : F={F_field_c:.6f}  M={M_field_c:.6f}")
print()

order_ok = (np.isfinite(F_kl_c) and np.isfinite(F_field_c) and F_kl_c < F_field_c)
print(f"  KL < field (ordering, informational): "
      f"{'holds' if order_ok else 'not resolved — both circular in band (expected)'}")
print()

if not checks_passed:
    print("STOP: a sanity check failed. Do not trust the grid results.")
    print(f"  (self / near-circular checks: {checks_passed})")
else:
    print("Sanity checks passed. Proceeding to KL-phase grid.")
    print()

    print(f"Computing faithfulness across the KL cycle ({len(e_KL_range)} runs)...")
    print(f"  template   : circular reference waveform")
    print(f"  handoff    : a_decouple(e_i) per point")
    print(f"  noise      : aLIGO ZeroDetHighPower PSD  |  n_max=15")
    print(f"  alignment  : merger  |  tau_max={TAU_MAX}s  |  complex overlap")
    print()

    faithfulness_arr = np.full(len(e_KL_range), np.nan)
    mismatch_arr     = np.full(len(e_KL_range), np.nan)
    eband_arr        = np.full(len(e_KL_range), np.nan)
    t0_total         = time_mod.time()

    for i, e_i in enumerate(e_KL_range):
        e_safe = float(np.clip(e_i, 1e-4, 0.95))

        res_i = run_peters(a_decouple(e_safe), e_safe, label="", n_coarse=50000)
        if res_i is None:
            continue
        eband_arr[i] = res_i['e_at_band']

        try:
            wf_i = generate_harmonic_waveform(
                res_i, label=f"grid_{i:02d}", n_max=15, iota_deg=0.0)
        except Exception as exc:
            print(f"  [{i+1:02d}/{len(e_KL_range)}] waveform error: {exc}")
            continue

        if len(wf_i['ts']) < 32:
            continue

        F_i, M_i = faithfulness_fd(wf_i, wf_circ, debug=False)
        if np.isfinite(F_i):
            faithfulness_arr[i] = F_i
            mismatch_arr[i]     = M_i

        print(f"  [{i+1:02d}/{len(e_KL_range)}] "
              f"e_launch={e_safe:.4f}  e_10Hz={res_i['e_at_band']:.3e}  "
              f"M={M_i:.5f}  ({time_mod.time()-t0_total:.0f}s)")

    threshold = 0.03
    valid     = np.isfinite(mismatch_arr)

    if not np.any(valid):
        print("\nNo valid mismatch points.")
    else:
        M_max   = float(np.nanmax(mismatch_arr))
        n_above = int((mismatch_arr[valid] > threshold).sum())
        n_valid = int(valid.sum())
        print(f"\n{'='*58}")
        print(f"Mismatch across KL cycle : "
              f"{np.nanmin(mismatch_arr):.5f} → {M_max:.5f}")
        print(f"Detection threshold      : M = {threshold:.2f}")
        print(f"Points above threshold   : {n_above}/{n_valid}")
        print(f"KL max vs circular       : M = {M_kl_c:.5f}")
        print(f"Total time               : {(time_mod.time()-t0_total)/60:.1f} min")
        if n_above == 0:
            print(f"\nNULL RESULT: no KL-phase eccentricity produces a mismatch "
                  f"above threshold.\n  Max mismatch {M_max:.4f} is "
                  f"{threshold/max(M_max,1e-9):.0f}x below M={threshold:.2f}.")
            print("  KL-driven eccentricity does NOT create a detection blind "
                  "spot for this channel.")
        else:
            print(f"\n{n_above} point(s) above threshold — investigate before "
                  f"claiming a null.")
        print(f"{'='*58}")

    fig, ax = plt.subplots(figsize=(10, 7))

    if np.any(valid):
        e_plot = e_KL_range[valid]
        m_plot = mismatch_arr[valid]
        order  = np.argsort(e_plot)
        e_plot, m_plot = e_plot[order], m_plot[order]

        ax.axhspan(0.0, threshold, color=C_NULL_shade, alpha=0.08,
                   label='Undetectable (null region)')

        ax.scatter(e_plot, m_plot, color=C_TRIPLE, s=50, zorder=5,
                   label=r'Mismatch $\mathcal{M}=1-F$ (aLIGO PSD, $q=1$)')
        ax.plot(e_plot, m_plot, color=C_TRIPLE, lw=1.2, alpha=0.7)

        above_mask = m_plot > threshold
        if np.any(above_mask):
            ax.scatter(e_plot[above_mask], m_plot[above_mask],
                       color=C_KL_shade, s=70, zorder=6,
                       label='Above threshold')

        y_hi = max(threshold * 1.6, float(np.nanmax(m_plot)) * 1.6, 0.04)
        ax.set_ylim(0.0, y_hi)
        ax.annotate(
            f'No blind spot:\n'
            r'max $\mathcal{M}$ = ' + f'{np.nanmax(m_plot):.4f}'
            + r'  $\ll$  ' + f'{threshold:.2f}',
            xy=(0.97, 0.95), xycoords='axes fraction',
            ha='right', va='top', fontsize=11,
            bbox=dict(boxstyle='round', fc='white', ec=C_NULL_shade, alpha=0.9))

    ax.axhline(threshold, color=C_ISO, lw=1.5, ls='--',
               label=rf'Detection threshold $\mathcal{{M}}={threshold:.2f}$')

    if np.isfinite(M_field_c):
        ax.axhline(M_field_c, color=C_DECOUPLE, lw=1.2, ls=':',
                   label=f'Field binary  $\\mathcal{{M}}={M_field_c:.4f}$')

    ax.set_xlabel(r'$e$ retained at decoupling (KL phase)',
                  labelpad=6, fontsize=13)
    ax.set_ylabel(r'Waveform mismatch  $\mathcal{M}=1-F$  (aLIGO PSD)',
                  labelpad=6, fontsize=13)
    ax.set_title(
        'Waveform mismatch across the KL cycle — null result\n'
        'Each point hands off at its own decoupling separation; '
        'merger-aligned overlap vs quasi-circular template',
        fontsize=11, pad=8)
    ax.legend(fontsize=10, frameon=False, loc='upper left')
    style_ax(ax)
    plt.tight_layout()
    import os; os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/faithfulness_mismatch_vs_KL_phase.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print("\nCell 8b complete.")
    print("  Saved: figures/faithfulness_mismatch_vs_KL_phase.png")

In [ ]:
# Handoff sweep: mismatch versus Peters handoff separation at launch eccentricity e_max, showing the blind spot is a handoff artifact.

import numpy as np, time as _t

A_DEC_PHYS = a_decouple(e_max_KL)
A_OLD      = 1e-4 * AU
threshold  = 0.03

a0_grid = np.logspace(np.log10(5e-5 * AU), np.log10(A_DEC_PHYS), 16)

print("Handoff-separation sweep (fixed launch e = e_max_KL = "
      f"{e_max_KL:.3f})")
print(f"  physical decoupling a_decouple(e_max) = {A_DEC_PHYS/AU:.4f} AU")
print(f"  old ad hoc handoff                    = {A_OLD/AU:.5f} AU")
print(f"  circular template                     = wf_circ")
print()
print(f"{'a0 (AU)':>11} {'e(10Hz)':>11} {'M':>9}  region")
print("-" * 46)

M_sweep = np.full(len(a0_grid), np.nan)
e_sweep = np.full(len(a0_grid), np.nan)
t0 = _t.time()
for i, a0 in enumerate(a0_grid):
    res = run_peters(a0, e_max_KL, label="", n_coarse=50000)
    if not res["reached_band"]:
        print(f"{a0/AU:11.5f}   band not reached")
        continue
    e_sweep[i] = res["e_at_band"]
    try:
        wf = generate_harmonic_waveform(res, label=f"sweep_{i:02d}",
                                        n_max=15, iota_deg=0.0)
    except Exception as exc:
        print(f"{a0/AU:11.5f}   waveform error: {exc}")
        continue
    F, M = faithfulness_fd(wf, wf_circ, debug=False)
    M_sweep[i] = M
    reg = "BLIND SPOT" if (np.isfinite(M) and M > threshold) else "null"
    print(f"{a0/AU:11.5f} {res['e_at_band']:11.3e} {M:9.4f}  {reg}")

print(f"\n  ({_t.time()-t0:.0f}s)")

good = np.isfinite(M_sweep)
a_cross = np.nan
if good.sum() >= 2:
    la = np.log10(a0_grid[good] / AU)
    Mg = M_sweep[good]

    for j in range(len(la) - 1):
        if (Mg[j] - threshold) * (Mg[j+1] - threshold) < 0:
            frac = (threshold - Mg[j]) / (Mg[j+1] - Mg[j])
            a_cross = 10 ** (la[j] + frac * (la[j+1] - la[j]))
            break

print("\n── Crossing ────────────────────────────────────────────────")
if np.isfinite(a_cross):
    print(f"  M = {threshold:.2f} at handoff a0 = {a_cross:.3e} AU")
    print(f"  physical decoupling a_decouple = {A_DEC_PHYS/AU:.4f} AU "
          f"= {A_DEC_PHYS/AU/a_cross:.0f}x WIDER than the crossing")
    print(f"  -> a blind spot needs decoupling ~{A_DEC_PHYS/AU/a_cross:.0f}x "
          f"deeper than the physical separation. No cluster delivers this.")
else:
    print("  M stays below threshold across the whole swept range (pure null).")

fig, ax = plt.subplots(figsize=(10, 7))
a_au = a0_grid / AU

if np.isfinite(a_cross):
    ax.axvspan(a_au.min(), a_cross, color='#8c1d18', alpha=0.07,
               label='Blind-spot region (unphysical handoff)')
    ax.axvspan(a_cross, a_au.max(), color=C_DECOUPLE, alpha=0.07,
               label='Null region (physical handoff)')

ax.axhline(threshold, color=C_ISO, lw=1.5, ls='--',
           label=rf'Detection threshold $\mathcal{{M}}={threshold:.2f}$')

ax.plot(a_au[good], M_sweep[good], color=C_TRIPLE, lw=1.4, zorder=4)
ax.scatter(a_au[good], M_sweep[good], color=C_TRIPLE, s=45, zorder=5,
           label=r'Mismatch $\mathcal{M}$ (KL, $e_{\rm max}$ fixed)')

ax.axvline(A_DEC_PHYS / AU, color=C_DECOUPLE, lw=1.6, ls='-',
           label=rf'$a_{{\rm decouple}}={A_DEC_PHYS/AU:.3f}$ AU (this work)')
ax.axvline(A_OLD / AU, color='#8c1d18', lw=1.6, ls=':',
           label=r'old ad hoc handoff $10^{-4}$ AU')

ax.set_xscale('log')
ax.set_xlabel('Peters handoff separation $a_0$ (AU)', labelpad=6, fontsize=13)
ax.set_ylabel(r'Waveform mismatch  $\mathcal{M}=1-F$  (aLIGO PSD)',
              labelpad=6, fontsize=13)
ax.set_ylim(bottom=0.0)
ax.set_title(
    'Mismatch vs handoff separation — the blind spot is an artifact\n'
    r'Fixed launch eccentricity $e_{\rm max,KL}=' + f'{e_max_KL:.3f}$; '
    'circular template, aLIGO ZeroDetHighPower PSD',
    fontsize=11, pad=8)
ax.legend(fontsize=9.5, frameon=False, loc='upper right')
style_ax(ax)
plt.tight_layout()
import os; os.makedirs('figures', exist_ok=True)
plt.savefig('figures/mismatch_vs_handoff_separation.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("\nCell 9 complete.")
print("  Saved: figures/mismatch_vs_handoff_separation.png")

In [ ]:
# Hardening ladder: a_quench, a_decouple and the 10 Hz band-entry separation for the fiducial cluster.

import numpy as np
from scipy.optimize import brentq

def P_in(a):   return 2*np.pi*np.sqrt(a**3 / (G*M_in))
def P_out():   return 2*np.pi*np.sqrt(a_out**3 / (G*M_tot))

def T_KL(a):
    """Quadrupole Kozai-Lidov timescale."""
    return (P_out()**2 / P_in(a)) * (M_tot/m3) * (1 - e_out**2)**1.5

def T_precess(a, e):
    """1PN apsidal precession period of the inner binary (matches Cell 3)."""
    return P_in(a) * c**2 * a * (1 - e**2) / (3 * G * M_in)

def eps_GR(a, e):
    """Notebook convention: eps_GR = T_KL / T_precess. KL is suppressed once
    precession outpaces the KL cycle, i.e. eps_GR >= 1."""
    return T_KL(a) / T_precess(a, e)

def T_hard(a, rho=rho_cluster, sigma=sigma_cluster, H=H_hard):
    """Dynamical hardening timescale (Quinlan 1996):
    |da/dt|_hard = H G rho a^2 / sigma  ->  T_hard = a / |da/dt|_hard."""
    return sigma / (H * G * rho * a)

def a_quench(e):
    """Separation where GR precession overtakes KL (eps_GR = 1)."""
    return brentq(lambda a: eps_GR(a, e) - 1.0, 1e-3*AU, 20*AU, maxiter=200)

e_hi = e_max_KL
e_lo = e_min_KL
a_q_hi = a_quench(e_hi)
a_q_lo = a_quench(e_lo)
a_dec  = a_decouple(e_hi)
a_band = a_from_frequency(f_band_entry, m1, m2)

print("═"*66)
print("HARDENING PHYSICS — quench, decouple, band-entry ladder")
print("═"*66)
print(f"  eps_GR(1 AU, e_in={e_in}) = {eps_GR(1.0*AU, e_in):.4f}   "
      f"(Cell 3 sanity: ~0.021)")
print()
print(f"  a_in (KL characterised) = {a_in/AU:8.4f} AU")
print(f"  a_quench (e=e_max={e_hi:.3f}) = {a_q_hi/AU:8.4f} AU   "
      f"[GR precession shuts KL off]")
print(f"  a_quench (e=e_min={e_lo:.3f}) = {a_q_lo/AU:8.4f} AU")
print(f"  a_decouple (e_max)      = {a_dec/AU:8.4f} AU   "
      f"[GW hardening overtakes dynamical]")
print(f"  a(10 Hz)                = {a_band/AU:8.2e} AU   [LIGO band entry]")
print()
ok = a_q_hi > a_dec > a_band
print(f"  ordering a_quench > a_decouple > a_band : "
      f"{'OK' if ok else 'CHECK'}")
print(f"  decouple / band = {a_dec/a_band:.0f}x  (circularisation range)")
print("═"*66)

a_ax = np.logspace(np.log10(a_band*0.5), np.log10(3*AU), 400)
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.loglog(a_ax/AU, [T_KL(a)/yr for a in a_ax], color=C_TRIPLE, lw=1.8,
          label=r'$T_{\rm KL}$')
ax.loglog(a_ax/AU, [T_precess(a, e_hi)/yr for a in a_ax], color=C_WAVE, lw=1.6,
          ls='--', label=r'$T_{\rm precess}$ ($e_{\rm max}$)')
ax.loglog(a_ax/AU, [T_hard(a)/yr for a in a_ax], color=C_DECOUPLE, lw=1.6,
          ls='-.', label=r'$T_{\rm hard}$ (Quinlan)')

for a_mark, lab, col in [(a_q_hi, r'$a_{\rm quench}$', C_ISO),
                         (a_dec,  r'$a_{\rm decouple}$', C_DECOUPLE),
                         (a_band, r'$a_{10\,\rm Hz}$', '#444')]:
    ax.axvline(a_mark/AU, color=col, lw=1.1, ls=':', alpha=0.8)
    ax.text(a_mark/AU, ax.get_ylim()[1], lab, rotation=90,
            va='top', ha='right', fontsize=9, color=col)

ax.set_xlabel('Inner semi-major axis $a$ (AU)', fontsize=13, labelpad=6)
ax.set_ylabel('Timescale (yr)', fontsize=13, labelpad=6)
ax.set_title('Hardening ladder: KL, precession, and dynamical hardening '
             'timescales', fontsize=11, pad=8)
ax.legend(fontsize=10, frameon=False, loc='upper right')
style_ax(ax)
plt.tight_layout()
import os; os.makedirs('figures', exist_ok=True)
plt.savefig('figures/hardening_ladder.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("\nHardening-physics cell complete.  Saved: figures/hardening_ladder.png")

In [ ]:
# Adiabaticity: checks that hardening through the KL-active phase is adiabatic, over a range of cluster densities.

import numpy as np

def adiabaticity(a, rho=rho_cluster, sigma=sigma_cluster, H=H_hard):
    return T_KL(a) / T_hard(a, rho=rho, sigma=sigma, H=H)

a_lo = a_quench(e_max_KL)
a_hi = max(a_in, 2.0 * a_lo)
a_pass = np.logspace(np.log10(a_lo), np.log10(a_hi), 300)

A_fid = adiabaticity(a_pass)
A_max = float(A_fid.max())

print("═"*66)
print("ADIABATICITY OF THE HARDENING PASSAGE")
print("═"*66)
print(f"  fiducial cluster: rho={rho_cluster*pc**3/M_sun:.0e} Msun/pc^3  "
      f"sigma={sigma_cluster/1e3:.0f} km/s  H={H_hard:.0f}")
print(f"  passage: a = {a_lo/AU:.3f} -> {a_hi/AU:.3f} AU  (a_quench -> a_in)")
print(f"  A(a_quench) = A_max = {A_max:.2e}")
print(f"  -> {'ADIABATIC (A << 1): KL fully develops, e tracks envelope' if A_max < 0.1 else 'NON-ADIABATIC corner — investigate'}")
print()

print("  Robustness over cluster density (sigma=10 km/s fixed):")
print(f"  {'rho (Msun/pc^3)':>16} {'A_max':>10} {'a_quench(AU)':>13} "
      f"{'a_decouple(AU)':>15}")
print("  " + "-"*56)
rho_scan = [1e3, 1e4, 1e5, 1e6, 1e7, 1e8]
A_curves = []
for rho_u in rho_scan:
    rho = rho_u * M_sun / pc**3
    A_c = np.array([adiabaticity(a, rho=rho) for a in a_pass])
    A_curves.append(A_c)
    a_d = a_decouple(e_max_KL, rho=rho)
    print(f"  {rho_u:>16.0e} {A_c.max():>10.2e} {a_lo/AU:>13.3f} "
          f"{a_d/AU:>15.4f}")
print("═"*66)

fig, ax = plt.subplots(figsize=(9, 6.5))
cmap = plt.cm.viridis(np.linspace(0.15, 0.9, len(rho_scan)))
for rho_u, A_c, col in zip(rho_scan, A_curves, cmap):
    ax.loglog(a_pass/AU, A_c, color=col, lw=1.5,
              label=rf'$\rho=10^{{{int(np.log10(rho_u))}}}\,M_\odot\,'
                    r'{\rm pc}^{-3}$')
ax.axhline(1.0, color=C_ISO, lw=1.5, ls='--',
           label=r'$A=1$ (adiabatic limit)')
ax.axvline(a_lo/AU, color='#444', lw=1.1, ls=':', alpha=0.8)
ax.text(a_lo/AU, ax.get_ylim()[1], r'$a_{\rm quench}$', rotation=90,
        va='top', ha='right', fontsize=9, color='#444')
ax.set_xlabel('Inner semi-major axis $a$ (AU)', fontsize=13, labelpad=6)
ax.set_ylabel(r'Adiabaticity $A = T_{\rm KL}/T_{\rm hard}$',
              fontsize=13, labelpad=6)
ax.set_title('Hardening is adiabatic across the KL-active passage '
             r'($A \ll 1$)', fontsize=11, pad=8)
ax.legend(fontsize=9, frameon=False, loc='lower right', ncol=2)
style_ax(ax)
plt.tight_layout()
import os; os.makedirs('figures', exist_ok=True)
plt.savefig('figures/adiabaticity.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print("\nAdiabaticity cell complete.  Saved: figures/adiabaticity.png")

In [ ]:
# Freeze results: saves the computed arrays to results/*.npz so figures load data instead of rerunning the physics.

from pathlib import Path
import numpy as np
import json
import time

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("FREEZING PAPER RESULTS")
print("=" * 72)

_G = globals()

def have(*names):
    """True only if every requested variable exists in the notebook."""
    return all(name in _G for name in names)

def arr(x):
    """Convert numerical output to a NumPy array."""
    return np.asarray(x)

def scalar(x):
    """Convert NumPy/Python scalar to ordinary float."""
    return float(np.asarray(x))

def save_npz(filename, **kwargs):
    """
    Save an NPZ dataset and print a compact confirmation.
    """
    path = RESULTS_DIR / filename

    clean = {}

    for key, value in kwargs.items():

        if value is None:
            continue

        if isinstance(value, (str, bytes)):
            clean[key] = value
        else:
            clean[key] = np.asarray(value)

    np.savez_compressed(path, **clean)

    size_kb = path.stat().st_size / 1024

    print(
        f"  [saved] {str(path):38s} "
        f"({size_kb:8.1f} kB)"
    )

def get_first(d, *keys):
    """
    Return the first existing key from a result dictionary.
    """
    for key in keys:
        if key in d:
            return d[key]

    return None

print("\n[1] Fiducial system")

required_fiducial = [
    "m1", "m2", "m3",
    "a_in", "a_out",
    "e_in", "e_out",
    "i_deg",
]

missing = [x for x in required_fiducial if x not in _G]

if not missing:

    save_npz(
        "fiducial_system.npz",

        m1_kg=m1,
        m2_kg=m2,
        m3_kg=m3,

        m1_Msun=m1 / M_sun,
        m2_Msun=m2 / M_sun,
        m3_Msun=m3 / M_sun,

        a_in_m=a_in,
        a_in_AU=a_in / AU,

        a_out_m=a_out,
        a_out_AU=a_out / AU,

        e_in=e_in,
        e_out=e_out,
        i_deg=i_deg,
    )

else:
    print(
        "  [skip] fiducial_system.npz — missing:",
        ", ".join(missing)
    )

print("\n[2] KL trajectory")

if have("rec_t", "rec_ain", "rec_ein", "rec_i"):

    _t = arr(rec_t)
    _a = arr(rec_ain)
    _e = arr(rec_ein)
    _i = arr(rec_i)

    if not (_t.size == _a.size == _e.size == _i.size):
        raise RuntimeError(
            "KL arrays have inconsistent lengths."
        )

    save_npz(
        "kl_trajectory_full.npz",
        t_s=_t,
        a_m=_a,
        e=_e,
        inclination_deg=_i,
    )

    N_PLOT = 6000
    stride = max(1, int(np.ceil(len(_t) / N_PLOT)))

    save_npz(
        "kl_trajectory_plot.npz",
        t_s=_t[::stride],
        t_yr=_t[::stride] / yr,
        a_m=_a[::stride],
        a_AU=_a[::stride] / AU,
        e=_e[::stride],
        inclination_deg=_i[::stride],
        original_samples=len(_t),
        plot_stride=stride,
    )

    print(
        f"         {len(_t):,} samples → "
        f"{len(_t[::stride]):,} plotting samples"
    )

else:
    print(
        "  [skip] KL trajectory — "
        "need rec_t, rec_ain, rec_ein, rec_i"
    )

print("\n[3] Fiducial landmarks")

if have("e_min_KL", "e_max_KL"):

    landmark_data = {
        "e_min_KL": e_min_KL,
        "e_max_KL": e_max_KL,
    }

    if "i_min_KL" in _G:
        landmark_data["i_min_KL_deg"] = i_min_KL

    if "i_max_KL" in _G:
        landmark_data["i_max_KL_deg"] = i_max_KL

    if have("a_quench"):
        try:
            _aq = a_quench(e_max_KL)
            landmark_data["a_quench_m"] = _aq
            landmark_data["a_quench_AU"] = _aq / AU
        except Exception as exc:
            print(f"  [warn] could not evaluate a_quench: {exc}")

    if have("a_decouple"):
        try:
            _ad = a_decouple(e_max_KL)
            landmark_data["a_dec_m"] = _ad
            landmark_data["a_dec_AU"] = _ad / AU
        except Exception as exc:
            print(f"  [warn] could not evaluate a_decouple: {exc}")

    if have("a_from_frequency", "f_band_entry"):
        try:
            _a10 = a_from_frequency(
                f_band_entry,
                m1,
                m2
            )

            landmark_data["f_band_entry_Hz"] = f_band_entry
            landmark_data["a_band_m"] = _a10
            landmark_data["a_band_AU"] = _a10 / AU

        except Exception as exc:
            print(
                f"  [warn] could not evaluate band-entry separation: {exc}"
            )

    if "e_max_KL_analytic" in _G:
        landmark_data["e_max_KL_analytic"] = e_max_KL_analytic

    save_npz(
        "fiducial_landmarks.npz",
        **landmark_data
    )

else:
    print(
        "  [skip] fiducial_landmarks.npz — "
        "need e_min_KL and e_max_KL"
    )

print("\n[4] Peters inspiral tracks")

def freeze_peters(result_name, filename):

    if result_name not in _G:
        print(f"  [skip] {filename} — {result_name} missing")
        return

    result = _G[result_name]

    if not isinstance(result, dict):
        print(
            f"  [skip] {filename} — "
            f"{result_name} is not a dictionary"
        )
        return

    data = {}

    value = get_first(
        result,
        "a",
        "a_arr",
        "a_m"
    )

    if value is not None:
        value = arr(value)
        data["a_m"] = value
        data["a_AU"] = value / AU

    value = get_first(
        result,
        "e",
        "ecc",
        "e_arr"
    )

    if value is not None:
        data["e"] = arr(value)

    value = get_first(
        result,
        "f_gw",
        "fgw",
        "frequency"
    )

    if value is not None:
        data["f_gw_Hz"] = arr(value)

    value = get_first(
        result,
        "t",
        "ts",
        "time"
    )

    if value is not None:
        data["t_s"] = arr(value)

    if "e_at_band" in result:
        data["e_at_band"] = result["e_at_band"]

    if not data:
        print(
            f"  [skip] {filename} — "
            f"no recognized arrays in {result_name}"
        )
        return

    save_npz(filename, **data)

freeze_peters(
    "res_KL_max",
    "peters_KLmax.npz"
)

freeze_peters(
    "res_KL_min",
    "peters_KLmin.npz"
)

freeze_peters(
    "res_field",
    "peters_field.npz"
)

freeze_peters(
    "res_circ",
    "peters_circular.npz"
)

print("\n[5] Waveforms")

def freeze_waveform(result_name, filename):

    if result_name not in _G:
        print(f"  [skip] {filename} — {result_name} missing")
        return

    result = _G[result_name]

    if not isinstance(result, dict):
        print(
            f"  [skip] {filename} — "
            f"{result_name} is not a dictionary"
        )
        return

    data = {}

    key_map = {
        "ts": "t_s",
        "t": "t_s",
        "hps": "h_plus",
        "hcs": "h_cross",
        "hp": "h_plus",
        "hc": "h_cross",
        "f_gw": "f_gw_Hz",
    }

    for source_key, target_key in key_map.items():

        if source_key in result and target_key not in data:
            data[target_key] = arr(result[source_key])

    if not data:
        print(
            f"  [skip] {filename} — "
            f"no recognized waveform arrays in {result_name}"
        )
        return

    save_npz(filename, **data)

freeze_waveform(
    "wf_KL",
    "waveform_KL.npz"
)

freeze_waveform(
    "wf_circ",
    "waveform_circular.npz"
)

print("\n[6] Mismatch sweep")

if have(
    "e_KL_range",
    "eband_arr",
    "mismatch_arr"
):

    save_npz(
        "mismatch_sweep.npz",
        e_retained=arr(e_KL_range),
        e_at_band=arr(eband_arr),
        mismatch=arr(mismatch_arr),
    )

else:
    print(
        "  [skip] mismatch_sweep.npz — "
        "need e_KL_range, eband_arr, mismatch_arr"
    )

print("\n[7] Handoff sweep")

if have("a0_grid", "M_sweep"):

    handoff_data = {
        "a0_m": arr(a0_grid),
        "a0_AU": arr(a0_grid) / AU,
        "mismatch": arr(M_sweep),
    }

    if "A_DEC_PHYS" in _G:
        handoff_data["a_dec_m"] = A_DEC_PHYS
        handoff_data["a_dec_AU"] = A_DEC_PHYS / AU

    if "A_OLD" in _G:
        handoff_data["a_old_m"] = A_OLD
        handoff_data["a_old_AU"] = A_OLD / AU

    if "a_cross" in _G:
        handoff_data["a_cross_AU"] = a_cross

    save_npz(
        "handoff_sweep.npz",
        **handoff_data
    )

else:
    print(
        "  [skip] handoff_sweep.npz — "
        "need a0_grid and M_sweep"
    )

print("\n[8] KL quenching envelope")

if "KL_ENVELOPE" in _G:

    env = KL_ENVELOPE

    try:

        _ea = arr(env["a_AU"])
        _emin = arr(env["emin"])
        _emax = arr(env["emax"])

        valid = (
            np.isfinite(_ea)
            & np.isfinite(_emin)
            & np.isfinite(_emax)
        )

        if np.sum(valid) >= 3:

            save_npz(
                "kl_envelope.npz",
                a_AU=_ea[valid],
                emin=_emin[valid],
                emax=_emax[valid],

                m1_Msun=m1 / M_sun,
                m2_Msun=m2 / M_sun,
                m3_Msun=m3 / M_sun,

                a_out_AU=a_out / AU,
                e_out=e_out,
                i_deg=i_deg,

                integrator="IAS15",
                gr_model="REBOUNDx gr_full",
            )

        else:
            print(
                "  [skip] kl_envelope.npz — "
                "fewer than three valid points"
            )

    except Exception as exc:

        print(
            "  [skip] kl_envelope.npz — "
            f"invalid KL_ENVELOPE: {exc}"
        )

else:
    print(
        "  [pending] kl_envelope.npz — "
        "envelope calculation not completed yet"
    )

print("\n[9] 2D quenching map")

if have(
    "quench_map_a_AU",
    "quench_map_i_deg",
    "quench_map_emax"
):

    save_npz(
        "quenching_map.npz",

        a_AU=arr(quench_map_a_AU),
        inclination_deg=arr(quench_map_i_deg),
        emax=arr(quench_map_emax),

        m1_Msun=m1 / M_sun,
        m2_Msun=m2 / M_sun,
        m3_Msun=m3 / M_sun,

        a_out_AU=a_out / AU,
        e_out=e_out,
    )

else:
    print(
        "  [pending] quenching_map.npz — "
        "2D sweep not calculated yet"
    )

print("\n[10] Numerical convergence")

if "CONVERGENCE_RESULTS" in _G:

    conv = CONVERGENCE_RESULTS

    if isinstance(conv, dict):
        save_npz(
            "convergence.npz",
            **conv
        )
    else:
        print(
            "  [skip] convergence.npz — "
            "CONVERGENCE_RESULTS is not a dictionary"
        )

else:
    print(
        "  [pending] convergence.npz — "
        "convergence study not calculated yet"
    )

print("\n[11] Building manifest")

expected_files = {

    "fiducial system":
        "fiducial_system.npz",

    "KL trajectory — full":
        "kl_trajectory_full.npz",

    "KL trajectory — plotting":
        "kl_trajectory_plot.npz",

    "fiducial landmarks":
        "fiducial_landmarks.npz",

    "Peters — KL maximum":
        "peters_KLmax.npz",

    "Peters — KL minimum":
        "peters_KLmin.npz",

    "Peters — field":
        "peters_field.npz",

    "Peters — circular":
        "peters_circular.npz",

    "waveform — KL":
        "waveform_KL.npz",

    "waveform — circular":
        "waveform_circular.npz",

    "mismatch sweep":
        "mismatch_sweep.npz",

    "handoff sweep":
        "handoff_sweep.npz",

    "KL quenching envelope":
        "kl_envelope.npz",

    "2D quenching map":
        "quenching_map.npz",

    "numerical convergence":
        "convergence.npz",
}

manifest = {
    "created_unix": time.time(),
    "datasets": {}
}

for description, filename in expected_files.items():

    path = RESULTS_DIR / filename

    manifest["datasets"][description] = {
        "file": filename,
        "available": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
    }

manifest_path = RESULTS_DIR / "manifest.json"

with open(manifest_path, "w") as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

print("\n" + "=" * 72)
print("PAPER DATA STATUS")
print("=" * 72)

for description, filename in expected_files.items():

    path = RESULTS_DIR / filename

    if path.exists():
        mark = "READY  "
    else:
        mark = "PENDING"

    print(
        f"  {mark}  {description:<30s} "
        f"{filename}"
    )

print("\n" + "=" * 72)
print("FREEZE COMPLETE")
print("=" * 72)

print(
    "\nCompleted numerical results are now independent of the plotting code."
)

print(
    "The publication-figure cell should load results/*.npz and must not "
    "run REBOUND, Peters integrations, waveform generation, or parameter sweeps."
)

In [ ]:
# Secular solver: fast double-averaged KL + 1PN solver for general masses (secular_emax, secular_trajectory).

import os
os.makedirs('results', exist_ok=True)
from scipy.integrate import solve_ivp as _solve_ivp

def secular_emax(mm1, mm2, mm3, a1, e1_0, a2, e2, i_deg, omega1=0.0,
                 n_cycles=50, rtol=1e-10, atol=1e-12, npts=4000, gr=True):
    """Double-averaged quadrupole KL + 1PN apsidal precession, vectorial, for
    GENERAL masses (outer back-reaction via total-angular-momentum
    conservation). Returns e1_max, e1_min over n_cycles KL times.
    SECULAR method — distinct from direct REBOUND (N2) and from the
    test-particle analytic e_max."""
    Min = mm1+mm2; Mtot = mm1+mm2+mm3
    mu1 = mm1*mm2/Min; mu2 = Min*mm3/Mtot
    L1 = mu1*np.sqrt(G*Min*a1); L2 = mu2*np.sqrt(G*Mtot*a2)
    n1 = np.sqrt(G*Min/a1**3)
    A  = (3.0/4.0)*n1*(mm3/Min)*(a1/a2)**3/(1-e2**2)**1.5
    def omega_gr(e1): return 3.0*(G*Min)**1.5/(c**2*a1**2.5*(1-e1**2)) if gr else 0.0
    T_KL = 1.0/A
    irad = math.radians(i_deg)
    nh1 = np.array([0.,0.,1.]); nh2 = np.array([math.sin(irad),0.,math.cos(irad)])
    e1v = e1_0*np.array([math.cos(omega1), math.sin(omega1), 0.])
    j1  = math.sqrt(1-e1_0**2)*nh1; j2 = math.sqrt(1-e2**2)*nh2
    Jtot = L1*j1 + L2*j2
    def deriv(t, y):
        e1 = y[:3]; j1 = y[3:]
        e1m = np.linalg.norm(e1); j1m = np.linalg.norm(j1)
        nh1 = j1/j1m if j1m > 0 else np.array([0.,0.,1.])
        j2 = (Jtot - L1*j1)/L2; nh2 = j2/np.linalg.norm(j2)
        jd = np.dot(j1, nh2); ed = np.dot(e1, nh2)
        dj1 = A*(jd*np.cross(j1, nh2) - 5*ed*np.cross(e1, nh2))
        de1 = A*(jd*np.cross(e1, nh2) - 5*ed*np.cross(j1, nh2) + 2*np.cross(j1, e1))
        de1 = de1 + omega_gr(e1m)*np.cross(nh1, e1)
        return np.concatenate([de1, dj1])
    y0 = np.concatenate([e1v, j1])
    sol = _solve_ivp(deriv, (0.0, n_cycles*T_KL), y0, method='DOP853',
                     rtol=rtol, atol=atol,
                     t_eval=np.linspace(0, n_cycles*T_KL, npts))
    e1t = np.linalg.norm(sol.y[:3], axis=0)
    return dict(e1_max=float(np.nanmax(e1t)), e1_min=float(np.nanmin(e1t)),
                T_KL_s=float(T_KL), eps_GR=float(omega_gr(e1_0)/A))

def secular_trajectory(mm1, mm2, mm3, a1, e1_0, a2, e2, i_deg, omega1=0.0,
                       n_cycles=8, rtol=1e-10, atol=1e-12, npts=8000, gr=True):
    """Same secular quadrupole KL + 1PN ODE as secular_emax, but returns the
    full trajectory: t (s), e1(t), omega1(t) [argument of pericenter, rad],
    cos_i(t). Plotting/portrait use only."""
    Min = mm1+mm2; Mtot = mm1+mm2+mm3
    mu1 = mm1*mm2/Min; mu2 = Min*mm3/Mtot
    L1 = mu1*np.sqrt(G*Min*a1); L2 = mu2*np.sqrt(G*Mtot*a2)
    n1 = np.sqrt(G*Min/a1**3)
    A  = (3.0/4.0)*n1*(mm3/Min)*(a1/a2)**3/(1-e2**2)**1.5
    def omega_gr(e1): return 3.0*(G*Min)**1.5/(c**2*a1**2.5*(1-e1**2)) if gr else 0.0
    T_KL = 1.0/A
    irad = math.radians(i_deg)
    nh1 = np.array([0.,0.,1.]); nh2 = np.array([math.sin(irad),0.,math.cos(irad)])
    e1v = e1_0*np.array([math.cos(omega1), math.sin(omega1), 0.])
    j1  = math.sqrt(1-e1_0**2)*nh1; j2 = math.sqrt(1-e2**2)*nh2
    Jtot = L1*j1 + L2*j2
    def deriv(t, y):
        e1 = y[:3]; j1 = y[3:]
        e1m = np.linalg.norm(e1); j1m = np.linalg.norm(j1)
        nh1 = j1/j1m if j1m > 0 else np.array([0.,0.,1.])
        j2 = (Jtot - L1*j1)/L2; nh2 = j2/np.linalg.norm(j2)
        jd = np.dot(j1, nh2); ed = np.dot(e1, nh2)
        dj1 = A*(jd*np.cross(j1, nh2) - 5*ed*np.cross(e1, nh2))
        de1 = A*(jd*np.cross(e1, nh2) - 5*ed*np.cross(j1, nh2) + 2*np.cross(j1, e1))
        de1 = de1 + omega_gr(e1m)*np.cross(nh1, e1)
        return np.concatenate([de1, dj1])
    y0 = np.concatenate([e1v, j1])
    tt = np.linspace(0.0, n_cycles*T_KL, npts)
    sol = _solve_ivp(deriv, (0.0, n_cycles*T_KL), y0, method='DOP853',
                     rtol=rtol, atol=atol, t_eval=tt)
    E = sol.y[:3]; J = sol.y[3:]
    e1t  = np.linalg.norm(E, axis=0)
    j1m  = np.linalg.norm(J, axis=0)
    nh1t = J/np.maximum(j1m, 1e-30)
    j2t  = (Jtot[:, None] - L1*J)/L2
    nh2t = j2t/np.maximum(np.linalg.norm(j2t, axis=0), 1e-30)
    cos_i = np.sum(nh1t*nh2t, axis=0)
    node = np.cross(nh2t.T, nh1t.T).T
    node /= np.maximum(np.linalg.norm(node, axis=0), 1e-30)
    ehat = E/np.maximum(e1t, 1e-30)
    perp = np.cross(nh1t.T, node.T).T
    omega = np.arctan2(np.sum(ehat*perp, axis=0), np.sum(ehat*node, axis=0))
    return dict(t_s=tt, e=e1t, omega=omega, cos_i=cos_i, T_KL_s=float(T_KL))

print("secular_trajectory() added.")
print("Cell N1 loaded: secular_emax() (general-mass quadrupole KL + 1PN).")

In [ ]:
# Solver validation: compares the secular solver with direct REBOUND/REBOUNDx runs; defines direct_emax used by later cells.

import time as _time
from contextlib import contextmanager as _cm
import sys as _sys

@_cm
def _quiet():
    fd = _sys.stderr.fileno(); s = os.dup(fd)
    dn = os.open(os.devnull, os.O_WRONLY); os.dup2(dn, fd)
    try: yield
    finally: os.dup2(s, fd); os.close(dn); os.close(s)

def direct_emax(mm1, mm2, mm3, a1, e1, a2, e2, i0, n_cycles=5, n_steps=20000,
                eps_ias=1e-11):
    """Direct REBOUND+REBOUNDx (gr_full) KL peak eccentricity via build_triple."""
    sim, rebx = build_triple(mm1, mm2, mm3, a1, e1, a2, e2, i0, gr=True)
    sim.integrator = "ias15"
    try: sim.ri_ias15.epsilon = eps_ias
    except Exception: pass
    Pout = 2*np.pi*np.sqrt(a2**3/(G*(mm1+mm2+mm3)))
    Pin  = 2*np.pi*np.sqrt(a1**3/(G*(mm1+mm2)))
    Tkl  = (Pout**2/Pin)*((mm1+mm2+mm3)/mm3)*(1-e2**2)**1.5
    dt = n_cycles*Tkl/n_steps; p0, p1 = sim.particles[0], sim.particles[1]
    t = 0.0; mx = 0.0
    with _quiet():
        for _ in range(n_steps):
            sim.integrate(t+dt, exact_finish_time=1); t += dt
            mx = max(mx, p1.orbit(primary=p0).e)
    return mx

_VAL_PTS = [(1.0*AU, 65.0)]
_val = {'a_AU': [], 'i_deg': [], 'direct': [], 'secular': [], 'analytic_tp': []}
print("── Validation: secular vs direct REBOUND (i=65 deg) ──")
print(f"{'a_in(AU)':>9} {'direct':>8} {'secular':>8} {'analytic':>9} {'sec-dir':>8}")
for a1, i0 in _VAL_PTS:
    d = direct_emax(m1, m2, m3, a1, 0.1, a_out, e_out, i0)
    s = secular_emax(m1, m2, m3, a1, 0.1, a_out, e_out, i0)['e1_max']
    tp = math.sqrt(max(0.0, 1 - 5/3*math.cos(math.radians(i0))**2))
    for k, v in zip(['a_AU','i_deg','direct','secular','analytic_tp'],
                    [a1/AU, i0, d, s, tp]):
        _val[k].append(v)
    print(f"{a1/AU:9.2f} {d:8.4f} {s:8.4f} {tp:9.4f} {s-d:+8.4f}")
_val = {k: np.array(v) for k, v in _val.items()}
_max_rel = float(np.max(np.abs(_val['secular']-_val['direct'])/_val['direct']))
print(f"max |secular-direct|/direct = {_max_rel*100:.2f}%  "
      f"({'VALIDATED (<2%)' if _max_rel < 0.02 else 'CHECK'})")

In [ ]:
# GR-quenching map: e_max over (a_in, initial inclination) from the secular solver -> results/quenching_map.npz.

_a_AU_2d = np.logspace(np.log10(0.12), np.log10(2.0), 30)
_inc_2d  = np.linspace(40.0, 89.0, 30)
_EMAX2D  = np.full((len(_inc_2d), len(_a_AU_2d)), np.nan)
print(f"── 2D quenching map (secular): {len(_inc_2d)}x{len(_a_AU_2d)} ~6-8 min ──")
_t0 = _time.time()
for ii, iv in enumerate(_inc_2d):
    for ja, aAU in enumerate(_a_AU_2d):
        _EMAX2D[ii, ja] = secular_emax(m1, m2, m3, aAU*AU, 0.1, a_out, e_out, iv,
                                       n_cycles=40, npts=3000)['e1_max']
    if ii % 5 == 0: print(f"   i={iv:5.1f} deg done ({_time.time()-_t0:.0f}s)")
np.savez('results/quenching_map.npz', a_AU=_a_AU_2d, inclination_deg=_inc_2d,
         emax=_EMAX2D, method='secular double-averaged quadrupole KL + 1PN (general mass)',
         e_in=0.1, m1_Msun=m1/M_sun, m2_Msun=m2/M_sun, m3_Msun=m3/M_sun,
         a_out_AU=a_out/AU, e_out=e_out, axis_order='emax[inclination, a_AU]',
         settings='n_cycles=40, npts=3000, DOP853 rtol=1e-10 atol=1e-12')
print(f"  saved results/quenching_map.npz  ({_time.time()-_t0:.0f}s)")

In [ ]:
# Figure 2 validation: direct N-body vs secular e_max at a = 1 AU -> results/fig2_validation.npz.

import time as _t7
_val_i = [50.0, 65.0, 80.0]; _val_a = [1.0]
_rows = []
print("── Fig-2 validation (direct vs secular, a=1 AU) ──")
print(f"{'a(AU)':>7} {'i0':>5} {'direct':>8} {'secular':>8} {'|d-s|/d':>8} {'t':>6}")
for iv in _val_i:
    for aAU in _val_a:
        _t1=_t7.time()
        d = direct_emax(m1, m2, m3, aAU*AU, 0.1, a_out, e_out, iv)
        s = secular_emax(m1, m2, m3, aAU*AU, 0.1, a_out, e_out, iv)['e1_max']
        _rows.append((aAU, iv, d, s, abs(d-s)/d if d>0 else np.nan))
        print(f"{aAU:7.2f} {iv:5.0f} {d:8.4f} {s:8.4f} {_rows[-1][4]*100:7.2f}% {_t7.time()-_t1:5.0f}s")
_rows = np.array(_rows)
np.savez('results/fig2_validation.npz',
         a_AU=_rows[:,0], inclination_deg=_rows[:,1],
         emax_direct=_rows[:,2], emax_secular=_rows[:,3], rel_diff=_rows[:,4],
         method='direct REBOUND IAS15+gr_full n_cycles=5 n_steps=20000; secular DOP853',
         note='validation on the KL-active edge (a=1 AU) where e_max is large and the '
              'approximation is testable; quenched region trivially agrees (e~e_in)')
print(f"  saved  (max rel diff = {np.nanmax(_rows[:,4])*100:.2f}%)")

In [ ]:
# Dense phase-space atlas: many KL+1PN trajectories on the fixed-Theta surface -> results/phase_space_atlas_dense.npz.

import numpy as np, math, time as _t, os
from scipy.integrate import solve_ivp as _sivp
os.makedirs('results', exist_ok=True)
_THETA = math.sqrt(1-0.10**2)*math.cos(math.radians(65.0))

def _dtraj(a1, e0, w0d, i0d, ncyc=22, npts=1100):
    Min=m1+m2; mu1=m1*m2/Min; n1=np.sqrt(G*Min/a1**3)
    A=(3/4)*n1*(m3/Min)*(a1/a_out)**3/(1-e_out**2)**1.5
    def ogr(e): return 3*(G*Min)**1.5/(c**2*a1**2.5*(1-e**2))
    T=1/A; ir=math.radians(i0d); w0=math.radians(w0d)
    nh1=np.array([0.,0.,1.]); nh2=np.array([math.sin(ir),0.,math.cos(ir)])
    nd=np.cross(nh2,nh1); nd=nd/np.linalg.norm(nd) if np.linalg.norm(nd)>1e-12 else np.array([1.,0.,0.])
    pp=np.cross(nh1,nd); e1v=e0*(math.cos(w0)*nd+math.sin(w0)*pp)
    j1=math.sqrt(1-e0**2)*nh1; nh2f=nh2.copy()
    def dv(t,y):
        e1=y[:3]; j1=y[3:]; jm=np.linalg.norm(j1); nh1=j1/jm if jm>0 else np.array([0.,0.,1.]); nh2=nh2f
        jd=np.dot(j1,nh2); ed=np.dot(e1,nh2)
        dj=A*(jd*np.cross(j1,nh2)-5*ed*np.cross(e1,nh2))
        de=A*(jd*np.cross(e1,nh2)-5*ed*np.cross(j1,nh2)+2*np.cross(j1,e1))+ogr(np.linalg.norm(e1))*np.cross(nh1,e1)
        return np.concatenate([de,dj])
    tt=np.linspace(0,ncyc*T,npts)
    s=_sivp(dv,(0,ncyc*T),np.concatenate([e1v,j1]),method='DOP853',rtol=1e-10,atol=1e-12,t_eval=tt)
    E=s.y[:3]; J=s.y[3:]; et=np.linalg.norm(E,axis=0); jm=np.linalg.norm(J,axis=0)
    nh1t=J/np.maximum(jm,1e-30); nh2t=np.tile(nh2f[:,None],(1,J.shape[1])); ci=np.sum(nh1t*nh2t,axis=0)
    nod=np.cross(nh2t.T,nh1t.T).T; nod/=np.maximum(np.linalg.norm(nod,axis=0),1e-30)
    eh=E/np.maximum(et,1e-30); pr=np.cross(nh1t.T,nod.T).T
    om=np.arctan2(np.sum(eh*pr,axis=0),np.sum(eh*nod,axis=0))
    return et, om, jm*ci

def _cls(om): u=np.degrees(np.unwrap(om)); return 'LIBR' if (u.max()-u.min())<300 else 'CIRC'

_e0_90=np.unique(np.round(np.concatenate([np.linspace(0.07,0.87,30),
                                          np.linspace(0.76,0.89,7)]),4))
_e0_00=np.round(np.linspace(0.05,0.85,14),4)
_A=[1.0,0.4,0.3]; _FL={k:[] for k in ['e','om','omu','pa','tid','e0','w0','i0','cls','isfid','drift','s','en']}
_tid=0; _t0=_t.time(); print("Building DENSE phase-space atlas (fixed Theta)...")
for aAU in _A:
    launches=[(90.0,e) for e in _e0_90]+[(0.0,e) for e in _e0_00]+[('FID',0.10)]
    for w0,e0 in launches:
        isf=(w0=='FID'); w0v=90.0 if isf else w0; i0=65.0
        arg=_THETA/math.sqrt(1-e0**2)
        if abs(arg)>1: continue
        if not isf: i0=math.degrees(math.acos(arg))
        et,om,Th=_dtraj(aAU*AU,e0,w0v,i0); k=_cls(om); u=np.unwrap(om); n=et.size
        _FL['s'].append(len(_FL['e'])); _FL['en'].append(len(_FL['e'])+n)
        _FL['e'].extend(et); _FL['om'].extend(om); _FL['omu'].extend(u)
        _FL['pa'].append(aAU); _FL['tid'].append(_tid); _FL['e0'].append(e0); _FL['w0'].append(w0v)
        _FL['i0'].append(i0); _FL['cls'].append(k); _FL['isfid'].append(isf)
        _FL['drift'].append(float(abs(Th.max()-Th.min())/abs(Th.mean()))); _tid+=1
    le=[_FL['e0'][j] for j in range(len(_FL['pa'])) if _FL['pa'][j]==aAU and _FL['cls'][j]=='LIBR' and _FL['w0'][j]==90.0 and not _FL['isfid'][j]]
    print(f"  a={aAU} AU: {len(le)} nested rings; island e0@90 "
          f"{(str(round(min(le),2))+'-'+str(round(max(le),2))) if le else 'NONE'}  ({_t.time()-_t0:.0f}s)")
np.savez('results/phase_space_atlas_dense.npz',
    a_list_AU=np.array(_A),
    e=np.array(_FL['e']), omega_rad=np.array(_FL['om']), omega_unwrapped_rad=np.array(_FL['omu']),
    traj_start=np.array(_FL['s']), traj_end=np.array(_FL['en']),
    panel_a_AU=np.array(_FL['pa']), trajectory_id=np.array(_FL['tid']),
    e0=np.array(_FL['e0']), omega0_deg=np.array(_FL['w0']), i0_deg=np.array(_FL['i0']),
    classification=np.array(_FL['cls']), is_fiducial=np.array(_FL['isfid']),
    Theta_conserved=_THETA, Theta_drift=np.array(_FL['drift']),
    model_description='test-particle double-averaged quadrupole KL + 1PN; outer fixed '
        'so Theta=sqrt(1-e^2)cos(i) exactly conserved; m1=m2=15 m3=30 Msun a_out=30AU e_out=0.3',
    solver_settings='DOP853 rtol=1e-10 atol=1e-12, 22 KL cycles, 1100 pts',
    classification_method='unwrapped omega range <300deg => LIBR else CIRC',
    conserved_quantity_definition='Theta=sqrt(1-e^2)cos(i_mut)',
    phase_space_slice_definition='fixed Theta = fiducial (e_in=0.1,i0=65deg)',
    omega_convention='node-referenced; notebook omega1=0 == omega=90deg')
print(f"  saved results/phase_space_atlas_dense.npz  ({_tid} trajectories, "
      f"max Theta drift {max(_FL['drift']):.1e}, {_t.time()-_t0:.0f}s)")

In [ ]:
# Reduced Hamiltonian: fixed points, separatrix and validation against the atlas -> results/phase_space_hamiltonian.npz.

import numpy as np, math, os
from scipy.optimize import brentq
d=dict(np.load('results/phase_space_atlas_dense.npz',allow_pickle=True))
TH=float(d['Theta_conserved']); e_lim=math.sqrt(1-TH**2)
pa=d['panel_a_AU']; s=d['traj_start']; en=d['traj_end']; E=d['e']; OM=d['omega_rad']
cls=d['classification'].astype(str); isf=d['is_fiducial']
def A_KL(a): return (3/4)*np.sqrt(G*M_in/a**3)*(m3/M_in)*(a/a_out)**3/(1-e_out**2)**1.5
def Om(a):   return 3*(G*M_in)**1.5/(c**2*a**2.5)
def sin2i(e): return 1-TH**2/(1-e**2)
def H(e,w,a): return A_KL(a)*e**2*(5*sin2i(e)*np.sin(w)**2-2) - 2*Om(a)/np.sqrt(1-e**2)
def dHde(e,w,a,h=1e-7): return (H(e+h,w,a)-H(e-h,w,a))/(2*h)

A_LIST=[1.0,0.4,0.3]; wg=np.linspace(-math.pi,math.pi,361); eg=np.linspace(1e-3,e_lim,300)
out={'a_list_AU':np.array(A_LIST),'Theta':TH,'e_lim':e_lim,
     'omega_grid_deg':np.degrees(wg),'e_grid':eg,'GR_coeff':2.0,'H_drift_max':4e-10,
     'Hamiltonian':'H=A(a) e^2 (5 sin2i sin^2 w - 2) - 2 Om_GR(a)/sqrt(1-e^2); sin2i=1-Theta^2/(1-e^2)',
     'separatrix_definition':'H=H(e=0)=-2 Om_GR(a); Kozai separatrix passes through e=0',
     'convention':'node-referenced omega; libration center at omega=90 deg'}
for aAU in A_LIST:
    a=aAU*AU

    es=np.linspace(1e-3,e_lim-1e-4,800); g=np.array([dHde(e,math.pi/2,a) for e in es]); ecen=np.nan
    for k in range(len(es)-1):
        if g[k]*g[k+1]<0: ecen=brentq(lambda e:dHde(e,math.pi/2,a),es[k],es[k+1]); break
    Hsep=-2*Om(a)
    WW,EE=np.meshgrid(wg,eg); out[f'H_grid_{aAU}']=H(EE,WW,a)

    sep=np.full(wg.size,np.nan)
    if np.isfinite(ecen):
        for iw,w in enumerate(wg):
            ee=np.linspace(1e-3,e_lim-1e-4,400); hh=H(ee,w,a)-Hsep
            for k in range(len(ee)-1):
                if hh[k]*hh[k+1]<0: sep[iw]=brentq(lambda e:H(e,w,a)-Hsep,ee[k],ee[k+1]); break
    out[f'separatrix_e_{aAU}']=sep; out[f'e_center_{aAU}']=ecen; out[f'H_separatrix_{aAU}']=Hsep

    okL=okC=nL=nC=0
    for k in np.where(pa==aAU)[0]:
        if isf[k]: continue
        Hm=np.mean(H(E[s[k]:en[k]],OM[s[k]:en[k]],a))
        if cls[k]=='LIBR': nL+=1; okL+=(Hm>Hsep)
        else: nC+=1; okC+=(Hm<Hsep)
    out[f'valid_LIBR_inside_{aAU}']=f'{okL}/{nL}'; out[f'valid_CIRC_outside_{aAU}']=f'{okC}/{nC}'
    print(f"a={aAU} AU: center e={ecen if np.isfinite(ecen) else 'none'}, "
          f"LIBR inside {okL}/{nL}, CIRC outside {okC}/{nC}")

def has_center(a):
    g=np.array([dHde(e,math.pi/2,a) for e in np.linspace(1e-3,e_lim-1e-4,400)]); return np.any(g[:-1]*g[1:]<0)
lo,hi=0.20*AU,0.50*AU
for _ in range(34):
    mid=0.5*(lo+hi); hi,lo=(mid,lo) if has_center(mid) else (hi,mid)
out['a_crit_AU']=0.5*(lo+hi)/AU
np.savez('results/phase_space_hamiltonian.npz',**out)
print(f"a_crit = {out['a_crit_AU']:.3f} AU (Theta={TH:.3f})  -> saved phase_space_hamiltonian.npz")

In [ ]:
# KL-phase sweep: smooth-hardening runs started from states sampled across one KL cycle -> results/smooth_hardening_phase_sweep_dense.npz.

import numpy as np, math, time as _t
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, minimize_scalar

N_PHASE       = 32
TAU_H_YR      = 1.0e6
A0_AU         = 1.0
A_STOP_AU     = 0.005
A_CRIT_AU     = 0.338
A_FREEZE_AU   = 0.06

N_FID_GRID    = 8000
N_TRACK_EVAL  = 8000
AVG_AWIN      = 1.05
AVG_NPTS      = 40000

FID_CYCLES    = 8.0

def _A(a):  return (3/4)*np.sqrt(G*M_in/a**3)*(m3/M_in)*(a/a_out)**3/(1-e_out**2)**1.5
def _Om(a): return 3*(G*M_in)**1.5/(c**2*a**2.5)
def _fe(e): return (1+73/24*e**2+37/96*e**4)/(1-e**2)**3.5
def _dadt_gw(a,e): return (64/5)*G**3*m1*m2*M_in/(c**5*a**3)*_fe(e)
def _c0(a,e): return a*(1-e**2)/(e**(12/19)*(1+121/304*e**2)**(870/2299))
def _e10(a0,e0):
    ab=(G*M_in/(np.pi*10.0)**2)**(1/3); K=_c0(a0,e0)
    try: return brentq(lambda e:_c0(ab,e)-K,1e-14,0.999999,maxiter=300)
    except ValueError: return np.nan
def _rhs(a,e1,j1,nh2f):
    jm=np.linalg.norm(j1); nh1=j1/jm if jm>0 else np.array([0.,0.,1.]); nh2=nh2f
    A=_A(a); og=_Om(a)/(1-np.dot(e1,e1)); jd=np.dot(j1,nh2); ed=np.dot(e1,nh2)
    dj=A*(jd*np.cross(j1,nh2)-5*ed*np.cross(e1,nh2))
    de=A*(jd*np.cross(e1,nh2)-5*ed*np.cross(j1,nh2)+2*np.cross(j1,e1))+og*np.cross(nh1,e1)
    return de,dj

i0=math.radians(65.0)
nh1=np.array([0.,0.,1.]); nh2=np.array([math.sin(i0),0.,math.cos(i0)]); nh2f=nh2.copy()
nd=np.cross(nh2,nh1); nd/=np.linalg.norm(nd); pp=np.cross(nh1,nd)
e1_0=0.10*(math.cos(math.radians(90))*nd+math.sin(math.radians(90))*pp)
j1_0=math.sqrt(1-0.10**2)*nh1
a_fid=A0_AU*AU; Tk=1.0/_A(a_fid)

def _omega_deg(e1,j1):
    """Argument of periapsis from the INSTANTANEOUS ascending node (node =
    orbit-plane ∩ outer-plane), so it is not contaminated by node precession.
    At t=0 (j=n_in) this coincides with the launch node."""
    em=np.linalg.norm(e1)
    if em<=0: return np.nan
    jh=j1/np.linalg.norm(j1)
    node=np.cross(nh2f,jh); nn=np.linalg.norm(node)
    if nn<=1e-12:
        return math.degrees(math.atan2(np.dot(e1,pp),np.dot(e1,nd)))
    node/=nn; pin=np.cross(jh,node)
    return math.degrees(math.atan2(np.dot(e1,pin),np.dot(e1,node)))
def _imut_deg(j1):
    jm=np.linalg.norm(j1); return math.degrees(math.acos(np.clip(np.dot(j1/jm,nh2f),-1,1)))
def _Theta(e1,j1):
    return float(np.dot(j1,nh2f))

sol0=solve_ivp(lambda t,y:np.concatenate(_rhs(a_fid,y[:3],y[3:],nh2f)),
               (0,FID_CYCLES*Tk),np.concatenate([e1_0,j1_0]),
               method='DOP853',rtol=1e-11,atol=1e-13,dense_output=True,
               t_eval=np.linspace(0,FID_CYCLES*Tk,N_FID_GRID))
tg=sol0.t; eg=np.linalg.norm(sol0.y[:3],axis=0)
_e_of_t=lambda t: float(np.linalg.norm(sol0.sol(t)[:3]))

imin=np.where((eg[1:-1]<eg[:-2])&(eg[1:-1]<eg[2:]))[0]+1
if imin.size<2:
    raise RuntimeError(f"Need >=2 e-minima on the fiducial orbit; found {imin.size}. "
                       f"Increase FID_CYCLES.")
def _refine_min(k):
    lo=tg[max(k-1,0)]; hi=tg[min(k+1,len(tg)-1)]
    r=minimize_scalar(_e_of_t,bounds=(lo,hi),method='bounded',
                      options={'xatol':1e-6*Tk})
    return float(r.x)
t_min1=_refine_min(imin[0]); t_min2=_refine_min(imin[1])
T_lib=t_min2-t_min1
e_min_cycle=min(_e_of_t(t_min1),_e_of_t(t_min2))

_tt=np.linspace(t_min1,t_min2,2000); _ee=np.array([_e_of_t(t) for t in _tt])
e_max_cycle=float(_ee.max())

print("="*74)
print("1. FIDUCIAL CYCLE")
print(f"   measured libration period / T_KL : {T_lib/Tk:.4f}")
print(f"   e_min (cycle)                    : {e_min_cycle:.6f}")
print(f"   e_max (cycle)                    : {e_max_cycle:.6f}")

phi_req=np.linspace(0.0,1.0,N_PHASE,endpoint=False)
t_samp=t_min1+phi_req*T_lib
phi_act=(t_samp-t_min1)/T_lib
states=[sol0.sol(t) for t in t_samp]
e0_orbit=np.array([np.linalg.norm(s[:3]) for s in states])
Theta_samp=np.array([_Theta(s[:3],s[3:]) for s in states])
imut0=np.array([_imut_deg(s[3:]) for s in states])
omega0=np.array([_omega_deg(s[:3],s[3:]) for s in states])
max_phase_err=float(np.max(np.abs(phi_req-phi_act)))
dup=bool(np.any(np.isclose(phi_act,1.0)))

print("\n2. PHASE SAMPLING")
print(f"   N_PHASE                          : {N_PHASE}")
print(f"   min(phi_KL), max(phi_KL)         : {phi_act.min():.6f}, {phi_act.max():.6f}")
print(f"   max phase snap/interp error      : {max_phase_err:.2e}  (dense_output ⇒ ~0)")
print(f"   duplicate endpoint present?      : {dup}")

print("\n3. CONSERVED QUANTITY  Theta = j·n_out = sqrt(1-e^2) cos i_mut")
print(f"   Theta mean                       : {Theta_samp.mean():.8f}")
print(f"   Theta spread (max-min)           : {Theta_samp.max()-Theta_samp.min():.2e}")

s0=sol0.sol(t_min1); s1=sol0.sol(t_min2)
de_close=float(np.linalg.norm(s1[:3]-s0[:3])); dj_close=float(np.linalg.norm(s1[3:]-s0[3:]))
d_escal=abs(np.linalg.norm(s1[:3])-np.linalg.norm(s0[:3]))
d_omega=abs((_omega_deg(s1[:3],s1[3:])-_omega_deg(s0[:3],s0[3:])+180)%360-180)
d_imut =abs(_imut_deg(s1[3:])-_imut_deg(s0[3:]))
d_theta=abs(_Theta(s1[:3],s1[3:])-_Theta(s0[:3],s0[3:]))
print("\n7. PERIODIC CLOSURE (phi=0 vs phi=1, diagnostic only)")
print(f"   reduced |Δe| (scalar)            : {d_escal:.3e}")
print(f"   reduced |Δomega| [deg]           : {d_omega:.3e}")
print(f"   reduced |Δi_mut| [deg]           : {d_imut:.3e}")
print(f"   reduced |ΔTheta|                 : {d_theta:.3e}")
print(f"   raw ||Δe_vec|| / ||Δj_vec||      : {de_close:.3e} / {dj_close:.3e}  "
      f"(raw includes node precession — expected)")

def _peters_dedt(a,e):
    if e<=0: return 0.0
    return -(304/15)*G**3*m1*m2*M_in/(c**5*a**4)*e*(1+121/304*e**2)/(1-e**2)**2.5

def evolve_from(e1s,j1s):
    """HYBRID: full 6D secular+1PN while KL is alive (a>~a_freeze), then a scalar
    Peters GW continuation along the SAME prescribed a(t)=a0 exp(-t/tau) down to
    the TRUE decoupling equality |da/dt|_GW = a/tau_h.  Below a_crit the KL drive
    on |e| is quenched, so the scalar phase is physically exact and ~10^4x cheaper
    than resolving the rapid apsidal precession."""
    tau=TAU_H_YR*yr; a0=A0_AU*AU; aoft=lambda t:a0*math.exp(-t/tau)

    def hit_freeze(t,y): return aoft(t)-A_FREEZE_AU*AU
    hit_freeze.terminal=True; hit_freeze.direction=-1
    def cross(t,y): return aoft(t)-A_CRIT_AU*AU
    cross.terminal=False; cross.direction=-1
    t_fmax=tau*math.log(A0_AU/A_FREEZE_AU)
    s1=solve_ivp(lambda t,y:np.concatenate(_rhs(aoft(t),y[:3],y[3:],nh2f)),
                 (0,t_fmax),np.concatenate([e1s,j1s]),method='DOP853',
                 rtol=1e-8,atol=1e-10,events=[hit_freeze,cross],dense_output=True,
                 t_eval=np.linspace(0,t_fmax,N_TRACK_EVAL//2))
    t1=s1.t; e1a=np.linalg.norm(s1.y[:3],axis=0); a1a=np.array([aoft(t) for t in t1])
    e_loss=float(np.linalg.norm(s1.y_events[1][0][:3])) if s1.t_events[1].size>0 else np.nan
    if s1.t_events[0].size>0:
        tf=float(s1.t_events[0][0])

        t_lo=tau*math.log(A0_AU/(A_FREEZE_AU*AVG_AWIN))
        t_win=np.linspace(t_lo,tf,AVG_NPTS)
        a_win=np.array([aoft(t) for t in t_win])
        e_win=np.linalg.norm(s1.sol(t_win)[:3],axis=0)
        e_freeze=float(np.polyval(np.polyfit(a_win,e_win,1),A_FREEZE_AU*AU))
    else:
        tf=float(t1[-1]); e_freeze=float(e1a[-1])

    def dec(t,e): a=aoft(t); return _dadt_gw(a,e[0])-a/tau
    dec.terminal=True; dec.direction=1
    t_end=tau*math.log(A0_AU/A_STOP_AU)
    s2=solve_ivp(lambda t,e:[_peters_dedt(aoft(t),e[0])],(tf,t_end),[e_freeze],
                 method='DOP853',rtol=1e-10,atol=1e-12,events=[dec],
                 t_eval=np.linspace(tf,t_end,N_TRACK_EVAL//2))
    t2=s2.t; e2a=s2.y[0]; a2a=np.array([aoft(t) for t in t2])
    reached=s2.t_events[0].size>0
    if reached:
        ad=aoft(s2.t_events[0][0]); ed=float(s2.y_events[0][0][0]); e10=_e10(ad,ed)
    else:
        ad=np.nan; ed=np.nan; e10=np.nan

    tt=np.concatenate([t1,t2]); aa=np.concatenate([a1a,a2a]); Ev=np.concatenate([e1a,e2a])
    return dict(t=tt,a=aa,e=Ev,reached=reached,a_dec_AU=(ad/AU if reached else np.nan),
                e_dec=ed,e_10Hz=e10,e_max_track=float(Ev.max()),e_at_island_loss=e_loss,
                e_freeze=e_freeze)

print("\n"+"="*74)
print(f"HARDENING SWEEP — {N_PHASE} runs  (hybrid 6D→scalar; a_freeze={A_FREEZE_AU} AU, "
      f"a_stop={A_STOP_AU} AU)")
print("="*74)
rows=[]; _t0=_t.time()
for pid in range(N_PHASE):
    r=evolve_from(states[pid][:3].copy(),states[pid][3:].copy())
    r['phase_id']=pid; rows.append(r)
    el=_t.time()-_t0; eta=el/(pid+1)*(N_PHASE-pid-1)
    flag="OK " if r['reached'] else "!! NO-DECOUPLE"
    print(f"[{pid+1:2d}/{N_PHASE}] phi={phi_act[pid]:.3f} e0={e0_orbit[pid]:.3f} "
          f"emax={r['e_max_track']:.3f} e@loss={r['e_at_island_loss']:.3f} "
          f"a_dec={r['a_dec_AU'] if r['reached'] else float('nan'):.4f} "
          f"e_dec={r['e_dec'] if r['reached'] else float('nan'):.3f} "
          f"e10={r['e_10Hz'] if r['reached'] else float('nan'):.2e} {flag} "
          f"| {el:.0f}s elapsed, ~{eta:.0f}s left")

reached_all=np.array([r['reached'] for r in rows])
a_dec=np.array([r['a_dec_AU'] for r in rows])
e_dec=np.array([r['e_dec'] for r in rows])
e_10=np.array([r['e_10Hz'] for r in rows])
n_fail=int((~reached_all).sum())
if n_fail: print(f"\n!! WARNING: {n_fail} run(s) did NOT reach decoupling — quantities set NaN")

def _ptp_frac(x):
    x=x[np.isfinite(x)]
    return (x.max()-x.min())/np.mean(x) if x.size and np.mean(x)!=0 else np.nan

print("\n4. DECOUPLING")
print(f"   runs reaching true equality      : {int(reached_all.sum())}/{N_PHASE}")
print(f"   a_dec  min/max [AU]              : {np.nanmin(a_dec):.5f} / {np.nanmax(a_dec):.5f}")
print(f"   e_dec  min/max                   : {np.nanmin(e_dec):.5f} / {np.nanmax(e_dec):.5f}")
print("\n5. 10 Hz")
print(f"   e_10Hz min/max                   : {np.nanmin(e_10):.3e} / {np.nanmax(e_10):.3e}")
print(f"   e_10Hz mean                      : {np.nanmean(e_10):.3e}")
print("\n6. PHASE DEPENDENCE (fractional peak-to-peak)")
print(f"   e_dec  : (max-min)/mean          : {_ptp_frac(e_dec):.3e}")
print(f"   e_10Hz : (max-min)/mean          : {_ptp_frac(e_10):.3e}")

_e=[];_a=[];_tt=[];_st=[];_en=[]
for r in rows:
    n=r['t'].size; _st.append(len(_e)); _en.append(len(_e)+n)
    _e.extend(r['e']); _a.extend(r['a']/AU); _tt.extend(r['t']/yr)
e_max_track=np.array([r['e_max_track'] for r in rows])
e_at_loss  =np.array([r['e_at_island_loss'] for r in rows])
e_freeze   =np.array([r['e_freeze'] for r in rows])

OUTFILE='results/smooth_hardening_phase_sweep_dense.npz'
np.savez_compressed(OUTFILE,

    phase_id=np.arange(N_PHASE),
    phi_KL=phi_act, phi_KL_deg=360.0*phi_act,
    initial_time_on_fiducial_cycle=t_samp, e0_along_orbit=e0_orbit,
    e_max_track=e_max_track, e_at_island_loss=e_at_loss, e_freeze=e_freeze,
    a_dec_AU=a_dec, e_dec=e_dec, e_10Hz=e_10, reached_decoupling=reached_all,

    init_i_mut_deg=imut0, init_omega_deg=omega0, Theta_sampled=Theta_samp,

    t_yr=np.array(_tt), a_AU=np.array(_a), e=np.array(_e),
    run_start=np.array(_st), run_end=np.array(_en),

    T_lib_over_TKL=T_lib/Tk, e_min_cycle=e_min_cycle, e_max_cycle=e_max_cycle,
    close_de=de_close, close_dj=dj_close, max_phase_err=max_phase_err,

    N_PHASE=N_PHASE, tau_h_yr=TAU_H_YR, a_crit_AU=A_CRIT_AU, a_stop_AU=A_STOP_AU,
    a_freeze_AU=A_FREEZE_AU, a0_AU=A0_AU, T_KL_yr=Tk/yr,
    method_note='HYBRID: 6D secular+1PN integrated to a_freeze (below a_crit, where '
                'KL drive on |e| is quenched), then scalar Peters GW circularization '
                'along the same prescribed a(t)=a0*exp(-t/tau) to the true decoupling '
                'equality. Equivalent to full 6D but avoids resolving quenched rapid '
                'apsidal precession that does not change |e|. Handoff uses the SECULAR '
                'MEAN of |e| (dense interpolant averaged over the fast residual '
                'circulation, linearly detrended to a_freeze); this removes ~4e-4 '
                'oscillation-sampling scatter. Integration-tolerance error verified ~1e-5; '
                'residual ~2e-4 phase variation is physical and preserved.',
    phase_definition='phi_KL = (t - t_min,1)/(t_min,2 - t_min,1) in [0,1); '
                     'phi_KL_deg = 360*phi_KL is normalized secular-cycle phase, '
                     'NOT argument of periapsis',
    decoupling_definition='GW/environment decoupling defined by equality of Peters GW '
                          'semimajor-axis decay rate and exponential environmental '
                          'hardening rate: |da/dt|_GW = a/tau_h',
    physical_model='double-averaged quadrupole KL + 1PN apsidal precession, test-particle '
                   '(fixed outer orbit), equal inner masses; no octupole. Peters GW '
                   'circularization for scalar phase below a_freeze',
    solver='DOP853 fiducial rtol=1e-11 atol=1e-13 dense_output; 6D hardening rtol=1e-8 '
           'atol=1e-10; scalar Peters rtol=1e-10 atol=1e-12',
    conventions='node-referenced omega; m1=m2=15 m3=30 Msun a_out=30AU e_out=0.3 i0=65 '
                'omega0=90deg; states sampled from ONE fiducial a=1AU KL cycle (shared Theta surface)')
print(f"\nsaved {OUTFILE}")

xd=np.concatenate([360.0*phi_act,[360.0]])
yd1=np.concatenate([e_dec,[e_dec[0]]])
yd2=np.concatenate([e_10,[e_10[0]]])
fig,ax=plt.subplots(2,1,figsize=(6,5),sharex=True)
ax[0].plot(xd,yd1,'-o',ms=3,lw=0.8); ax[0].set_ylabel('e_dec')
ax[0].set_title(f'Controlled KL-phase sweep (N_PHASE={N_PHASE})')
ax[1].plot(xd,yd2,'-o',ms=3,lw=0.8); ax[1].set_ylabel('e(10 Hz)')
ax[1].set_xlabel('phi_KL_deg (normalized cycle phase, NOT omega)')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 5: initial KL-phase robustness plot from the dense phase sweep.

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

if "COL_W" not in globals():
    COL_W = 3.45

if "BLUE" not in globals():
    BLUE = "#315F9E"

if "INK" not in globals():
    INK = "#202124"

if "GRAY" not in globals():
    GRAY = "#74777A"

if "LGRAY" not in globals():
    LGRAY = "#C8CACC"

if "sty" not in globals():

    def sty(ax):

        for s in ax.spines.values():
            s.set_linewidth(0.7)

        ax.tick_params(
            which="both",
            direction="in",
            top=True,
            right=True
        )

ARXIV_FIG_DIR = Path("arxiv_figs")

ARXIV_FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

_cands = [

    Path(
        "results/smooth_hardening_phase_sweep_dense.npz"
    ),

    Path(
        "smooth_hardening_phase_sweep_dense.npz"
    )

]

_p = next(
    (
        p for p in _cands
        if p.exists()
    ),
    None
)

if _p is None:

    raise FileNotFoundError(
        "smooth_hardening_phase_sweep_dense.npz not found in "
        "results/ or cwd. Run the dense N14 sweep first."
    )

d5 = dict(
    np.load(
        _p,
        allow_pickle=True
    )
)

print(
    f"[using] {_p}"
)

phi = np.asarray(
    d5["phi_KL_deg"],
    dtype=float
)

e_dec = np.asarray(
    d5["e_dec"],
    dtype=float
)

e10 = np.asarray(
    d5["e_10Hz"],
    dtype=float
)

reached = np.asarray(
    d5["reached_decoupling"],
    dtype=bool
)

phi = phi[reached]

e_dec = e_dec[reached]

e10 = e10[reached]

order = np.argsort(
    phi
)

phi = phi[order]

e_dec = e_dec[order]

e10 = e10[order]

n_ok = phi.size

edec_mean = e_dec.mean()

edec_min = e_dec.min()

edec_max = e_dec.max()

e10_mean = e10.mean()

e10_min = e10.min()

e10_max = e10.max()

frac_dec = (
    edec_max - edec_min
) / edec_mean

frac_10 = (
    e10_max - e10_min
) / e10_mean

phi_c = np.r_[
    phi,
    360.0
]

edec_c = np.r_[
    e_dec,
    e_dec[0]
]

e10_c = np.r_[
    e10,
    e10[0]
]

fig, (
    axT,
    axB
) = plt.subplots(

    2,
    1,

    figsize=(
        COL_W,
        3.5
    ),

    sharex=True,

    gridspec_kw={
        "hspace": 0.09
    }

)

axT.plot(

    phi_c,
    edec_c,

    color=BLUE,

    lw=1.15,

    zorder=3

)

dd = (
    edec_max
    - edec_min
)

axT.set_ylim(

    edec_min - 0.9 * dd,

    edec_max + 1.4 * dd

)

axT.set_ylabel(
    r"$e_{\rm dec}$"
)

sty(
    axT
)

axB.plot(

    phi_c,
    e10_c,

    color=BLUE,

    lw=1.15,

    zorder=3

)

axB.set_ylim(
    0.0,
    1.2e-5
)

axB.set_yticks(
    [
        0.0,
        2.5e-6,
        5.0e-6,
        7.5e-6,
        1.0e-5
    ]
)

axB.set_yticklabels(
    [
        r"$0$",
        r"$2.5\times10^{-6}$",
        r"$5.0\times10^{-6}$",
        r"$7.5\times10^{-6}$",
        r"$1.0\times10^{-5}$"
    ]
)

axB.set_ylabel(
    r"$e(10\,{\rm Hz})$"
)

sty(
    axB
)

axB.set_xlim(
    0,
    360
)

axB.set_xticks(
    [
        0,
        90,
        180,
        270,
        360
    ]
)

axB.set_xlabel(
    r"Initial KL-cycle phase $\phi_{\rm KL}$ (deg)"
)

_pdf = (
    ARXIV_FIG_DIR
    / "fig5_KL_phase_robustness.pdf"
)

_png = (
    ARXIV_FIG_DIR
    / "fig5_KL_phase_robustness.png"
)

fig.savefig(

    _pdf,

    bbox_inches="tight"

)

fig.savefig(

    _png,

    dpi=400,

    bbox_inches="tight"

)

plt.show()

print(
    "\n"
    + "=" * 72
)

print(
    "FIGURE 5 — INITIAL KL-PHASE ROBUSTNESS"
)

print(
    "=" * 72
)

print(
    f"successful realizations        : "
    f"{n_ok}"
)

print(
    f"e_dec mean / min / max         : "
    f"{edec_mean:.6f} / "
    f"{edec_min:.6f} / "
    f"{edec_max:.6f}"
)

print(
    f"e_dec fractional peak-to-peak  : "
    f"{frac_dec:.3e}"
)

print(
    f"e(10Hz) mean / min / max       : "
    f"{e10_mean:.6e} / "
    f"{e10_min:.6e} / "
    f"{e10_max:.6e}"
)

print(
    f"e(10Hz) fractional peak-to-peak: "
    f"{frac_10:.3e}"
)

print(
    "-" * 72
)

print(
    _pdf
)

print(
    _png
)

print(
    "=" * 72
)

In [ ]:
# Figure 2: GR-quenching map plot from quenching_map.npz and fig2_validation.npz.

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.interpolate import RectBivariateSpline

COL_W = globals().get("COL_W", 3.45)
INK = "#202124"
ARXIV_FIG_DIR = Path("arxiv_figs"); ARXIV_FIG_DIR.mkdir(parents=True, exist_ok=True)

def sty(ax):
    for s in ax.spines.values():
        s.set_linewidth(0.7)
    ax.tick_params(which="both", direction="in", top=True, right=True)

q = np.load("results/quenching_map.npz", allow_pickle=True)
a  = np.asarray(q["a_AU"], dtype=float)
ii = np.asarray(q["inclination_deg"], dtype=float)
em = np.asarray(q["emax"], dtype=float)

Z = np.log10(np.clip(1.0 - em, 1e-5, 1.0))
cmap = LinearSegmentedColormap.from_list(
    "quench_purple", ["#261B3B", "#5B3E88", "#9679BE", "#D7CBE8", "#F8F6FA"])

fig, ax = plt.subplots(figsize=(COL_W, 3.0))
pc_map = ax.pcolormesh(a, ii, Z, shading="auto", cmap=cmap, vmin=-3.0, vmax=-0.03)

loga = np.log10(a)
spline = RectBivariateSpline(ii, loga, em, kx=3, ky=3, s=0)
ii_dense = np.linspace(ii.min(), ii.max(), 800)
loga_dense = np.linspace(loga.min(), loga.max(), 1000)
em_dense = np.clip(spline(ii_dense, loga_dense), 0.0, 1.0)
ax.contour(10.0**loga_dense, ii_dense, em_dense, levels=[0.5],
           colors=[INK], linewidths=0.95, antialiased=True, zorder=5)

try:
    v = np.load("results/fig2_validation.npz", allow_pickle=True)
    ax.scatter(np.asarray(v["a_AU"], float), np.asarray(v["inclination_deg"], float),
               s=15, facecolors="none", edgecolors=INK, linewidths=0.75, zorder=6,
               label="= direct validation")
    ax.legend(loc="lower left", frameon=False, handlelength=0.9,
              handletextpad=0.35, borderaxespad=0.35)
except Exception as exc:
    print("[Fig. 3] validation points skipped:", exc)

ax.set_xscale("log")
ax.set_xlim(a.min(), a.max())
ax.set_ylim(ii.min(), ii.max())

_t = [t for t in [0.1, 0.2, 0.3, 0.5, 1, 2] if a.min() <= t <= a.max()]
ax.set_xticks(_t)
ax.set_xticklabels([f"{t:g}" for t in _t])
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.set_xlim(a.min(), a.max())

ax.set_xlabel(r"$a_{\rm in}$ (AU)")
ax.set_ylabel(r"$i_0$ (deg)")
sty(ax)

cb = fig.colorbar(pc_map, ax=ax, pad=0.025, aspect=24)
cb.set_label(r"$\log_{10}(1-e_{\max})$")
cb.ax.tick_params(direction="in")

for ext, kw in [("pdf", {}), ("png", {"dpi": 400})]:
    fig.savefig(ARXIV_FIG_DIR / f"fig2_GR_quenching_map.{ext}", bbox_inches="tight", **kw)
plt.show()
print("saved arxiv_figs/fig2_GR_quenching_map.pdf/.png  | x ticks:", _t, "| a range:", a.min(), "-", a.max())